
# Maneuvers & Formations — API Reference

## Core Primitives

### TroopPath

A resolved hex route with cached metadata. Returned by all pathfinding functions.

```python
path = piece.pathfind(target_hex, grid, elevations, countries=None)

path.hexes       # list[int] — ordered hex indices
path.cost        # float — total movement cost
path.start       # first hex
path.end         # last hex
path.as_set      # set[int] — cached for fast membership checks
path.to_rules()  # list[int] — convert to Instruction values

bool(path)       # False if empty or single-hex (unreachable)
len(path)        # number of hexes in route
```

**Movement costs:** flat/downhill = 1.0, uphill = 2.0. Water and invalid hexes are impassable.

### Pathfinding Variants

```python
# Basic A→B shortest path (Dijkstra)
path = piece.pathfind(target_hex, grid, elevations, countries)

# Nearest hex with food_tier >= threshold
path = piece.pathfind_to_food(grid, elevations, food_tiers, min_tier=3, countries)

# Path AWAY from a threat — prefers high ground in the opposite hemisphere
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)

# Convenience: pathfind + convert to InstructionList in one call
il = piece.pathfind_to_rules(target, grid, elevations, countries)
```

### InstructionList

The rule program for a piece. Each rule is an `Instruction` enum value.

```python
il = InstructionList(rules=[...], cursor=0, patrol=False)

il.insert_rules(new_rules)   # insert at cursor, advance cursor
il.delete_at(idx)             # remove one rule, clamp cursor
il.set_cursor(idx)            # move cursor
il.toggle_patrol()            # flip loop mode
il.est_turns(speed)           # lower-bound turn count for sync
il.compact(piece_id)          # FastHTML badge strip
```

**Patrol mode:** when `patrol=True`, cursor wraps to 0 at end of rules.
When `patrol=False`, piece stops after the last instruction.

### Instruction Budget Rules

Per turn, each piece gets `move_strength` budget points:

| Instruction | Cost | Notes |
|-------------|------|-------|
| FORWARD | 1 (flat/downhill) or 2 (uphill) | Into water/edge: auto-rotate left free, up to 5×. Into occupied hex: forced PAUSE cost 1. |
| ROT_L, ROT_R | 0 (free) | |
| PAUSE | 1 | |
| DEFEND | 1 | |
| HARVEST | 1 | |
| GIVE | 1 | |
| SETTLE | 1 | |

When budget runs out mid-instruction, the turn ends. If the last instruction
was non-movement (PAUSE/DEFEND/HARVEST/GIVE/SETTLE), remaining budget is
idled out as implicit PAUSEs.

---

## Formations

### FormationType

```python
class FormationType(Enum):
    LINE   = "line"    # row abreast, one step behind leader
    WEDGE  = "wedge"   # V-shape, leader at point, wings fan back
    BOX    = "box"     # ring-1 around leader (overflow to ring-2 if >6)
    COLUMN = "column"  # single file directly behind leader
```

**Choosing a formation:**
- **COLUMN** — narrow passages, rivers, mountain paths
- **LINE** — broad front for harvesting or sight coverage
- **WEDGE** — aggressive advance, concentrates force at point of contact
- **BOX** — protect a VIP (queen, settler), all-around defense

### form_up

Assemble followers around the leader's **current position**.

```python
orders = squad.form_up(
    FormationType.WEDGE, grid, elevations,
    leader=None,          # auto-picks highest rank if omitted
    countries=None,
    food_tiers=None,      # enables harvest padding during sync
    sync=True,            # pad so all arrive same turn
)
# orders: {piece_id: InstructionList} — leader NOT included
```

**Important:** `form_up` does not move the leader. It positions followers
relative to where the leader already is. Compose with `march_to` or
`pathfind_to_rules` for the leader's own movement.

### march_to

Move entire squad to a destination, arriving in formation.

```python
orders = squad.march_to(
    destination_hex,
    FormationType.LINE, grid, elevations,
    leader=None,          # auto-picks highest rank
    countries=None,
    food_tiers=None,
    sync=True,
)
# orders: {piece_id: InstructionList} — leader IS included
```

Formation offsets are computed relative to **destination + leader's arrival
facing**, so the formation assembles correctly on arrival. Uses Hungarian
matching to optimally assign followers to slots.

---

## Tactical Maneuvers

### surround

Position squad members around a target hex in a ring.

```python
orders = squad.surround(
    target_hex, grid, elevations,
    mode=SurroundMode.GUARD,  # GUARD = face outward, FEED = face inward
    max_ring=3,
    countries=None,
    elevation_mult=0.005,     # sight bonus per elevation unit
)
```

**Modes:**
- `GUARD` — defensive perimeter; sight cones scan outward for threats
- `FEED` — supply hub; GIVE cones converge on center piece

Uses Hungarian matching. Respects `effective_sight` — pieces won't be placed
farther than they can see.

### retreat

Scatter squad away from a threat. Each piece picks its own best retreat hex
independently (no formation maintained during withdrawal).

```python
orders = squad.retreat(
    threat_hex, grid, elevations,
    distance=5,               # max rings to search
    countries=None,
    food_tiers=None,
    sync=True,
    elevation_weight=0.5,     # preference for high ground
)
```

All pieces face **away** from the threat on arrival. Typical follow-up:

```python
# After executing retreat orders...
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### retreat_path (standalone)

Single-piece retreat. Returns a `TroopPath`.

```python
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)
```

Runs one Dijkstra flood-fill within `distance` rings. Scores candidates by:
```
score = elevation × elevation_weight − path_cost
```
Only considers hexes in the **away hemisphere** (cube dot product < 0 with
the toward-direction). Falls back to cheapest reachable hex if cornered.

---

## Synchronization

### sync_paths

Pad instruction lists so all pieces finish on the same turn.

```python
synced = sync_paths(
    pieces,                   # all pieces referenced by orders
    orders,                   # {piece_id: InstructionList}
    end_hexes=None,           # {piece_id: final_hex} for harvest check
    food_tiers=None,          # enables productive padding
    harvest_threshold=3,      # min food tier for harvest padding
    min_turns=0,              # floor (use leader's turns as floor)
)
```

**Padding priority:**
1. If `food_tiers[end_hex] >= threshold` → `HARVEST, GIVE, HARVEST, GIVE...`
2. Otherwise → `PAUSE, PAUSE, PAUSE...`

Called automatically by `form_up`, `march_to`, `surround`, and `retreat`
when `sync=True`.

---

## Helper Functions

```python
# Direction from one hex toward another (returns 0–5)
facing = _facing_toward(grid, from_idx, to_idx)

# Opposite direction (180°)
facing = _facing_away(grid, from_idx, to_idx)

# Effective sight with elevation bonus
sight = piece.effective_sight(elevations, elevation_mult=0.005)

# Formation slot offsets in cube coordinates
offsets = _formation_offsets(FormationType.WEDGE, n_followers, leader_facing)
```

---

## Recommended Combos

### Escort a Queen to a Settlement Site

```python
# Queen paths to destination
queen.instructions = queen.pathfind_to_rules(site, grid, elevations)

# Escort forms up around queen, then marches together
orders = escort_squad.march_to(
    site, FormationType.BOX, grid, elevations,
    leader=queen, food_tiers=food_tiers,
)
for p in escort_squad.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
```

### Defend a Settlement

```python
orders = garrison.surround(
    settlement.location, grid, elevations,
    mode=SurroundMode.GUARD, max_ring=2,
)
```

### Supply Chain: Harvest + Deliver

```python
# Pawn finds food, paths there, harvests, returns to queen
food_path = pawn.pathfind_to_food(grid, elevations, food_tiers)
return_path = pawn.pathfind(queen.location, grid, elevations,
                            source=food_path.end)

rules = (food_path.to_rules(grid, pawn.facing)
         + [Instruction.HARVEST.value] * 3
         + return_path.to_rules(grid, _facing_toward(grid, food_path.end, queen.location))
         + [Instruction.GIVE.value])

pawn.instructions = InstructionList(rules, patrol=True)
```

### Retreat + Regroup

```python
# Phase 1: scatter away from threat
retreat_orders = squad.retreat(enemy_hex, grid, elevations, distance=6)
for p in squad.alive:
    if p.id in retreat_orders:
        p.instructions = retreat_orders[p.id]

# Phase 2 (after retreat executes): regroup in defensive line
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### Pincer: Two Squads from Opposite Sides

```python
# Left wing approaches from the west
left_orders = left_squad.march_to(
    west_of_target, FormationType.LINE, grid, elevations)

# Right wing approaches from the east
right_orders = right_squad.march_to(
    east_of_target, FormationType.LINE, grid, elevations)

# Sync both wings to arrive simultaneously
all_pieces = left_squad.alive + right_squad.alive
all_orders = {**left_orders, **right_orders}
synced = sync_paths(all_pieces, all_orders)
```

### Scout Patrol (Loop)

```python
# Knight patrols a triangle: forward, turn, forward, turn...
knight.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
], patrol=True)  # loops forever
```

---

## Design Notes

**Hungarian matching** (`scipy.optimize.linear_sum_assignment`) is used
everywhere pieces need optimal assignment to positions: surround, form_up,
march_to. Cost matrix is `pathfind_cost[piece, slot]`. Complexity O(n³)
but n is squad size (typically 3–8), so it's instant.

**Occupied set is deliberately omitted** from plan overlays. Two pieces
can path through the same hex in preview. This is intentional — plans are
pre-execution previews, and collision resolution happens during `simulate()`.

**`form_up` vs `march_to`:**
- `form_up` — positions followers around leader's CURRENT hex. Leader not included in orders.
- `march_to` — positions everyone around a DESTINATION hex. Leader IS included in orders.

**`sync_paths` is composable.** Any function that returns
`dict[str, InstructionList]` can be synced. You can also sync orders from
different sources by merging the dicts before calling `sync_paths`.

**`retreat` breaks formation intentionally.** Each piece finds its own
best escape hex. Regroup afterward with `form_up`. Trying to maintain
formation during retreat is expensive and usually tactically wrong.

**Elevation matters in two places:**
1. Movement cost: uphill = 2×
2. Sight range: `effective_sight = base_sight + elevation × 0.005`

**Food integration:** All sync functions accept `food_tiers`. When a
piece's end hex has good food, slack turns become HARVEST+GIVE cycles
instead of idle PAUSEs. This means early-arriving units productively
farm while waiting for slower friends.



# Pyomo × HexMagic — Integration Notes

## The Chocolate and Peanut Butter

**Chocolate:** Pyomo's declarative optimization — define *what* you want, not *how* to search for it.
**Peanut butter:** HexMagic's composable overlays, `@patch`, `@dataclass`, and FastHTML rendering.

The goal: describe a tactical problem (supply a marching queen, assign formation slots,
plan a retreat) as a **model**, solve it, and get back `{piece_id: InstructionList}` —
the same dict everything else in the system already speaks.

---

## Why Pyomo Over Raw scipy.milp

We already use `scipy.optimize.linear_sum_assignment` (Hungarian) for formation slot
assignment and `scipy.optimize.milp` is imported. These work great for single-matrix
problems. But the supply chain problem has **coupled decisions across time**:

- Who goes where (assignment) affects who can give food to whom (visibility)
- Food at turn t depends on food at turn t-1 (temporal coupling)
- The queen's position changes each turn (moving target)

With scipy.milp you'd hand-build the A matrix, b vector, and variable bounds.
With Pyomo you write:

```python
def food_balance(m, p, t):
    if t == 0: return m.food[p, 0] == initial_food[p]
    return m.food[p, t] == (m.food[p, t-1] - m.diet[p]
        + harvest_if_assigned(m, p, t)
        + received(m, p, t) - given(m, p, t))
m.food_bal = pyo.Constraint(m.P, m.T, rule=food_balance)
```

Same math. Readable. Debuggable. And Pyomo compiles it to the same solver format.

---

## Solver: HiGHS (already installed)

Pyomo needs a solver backend. HiGHS ships with scipy and handles MILP natively.
The bridge is the `appsi_highs` interface in Pyomo, or we can use:

```python
# Option A: appsi_highs (if pyomo detects it)
solver = pyo.SolverFactory('appsi_highs')

# Option B: write to .mps, call scipy.milp manually
# (fallback if appsi doesn't work)

# Option C: install glpk
# !apt-get install -y glpk-utils  # then SolverFactory('glpk')
```

Test which works first — that's the first cell of the new notebook.

---

## The Three Optimization Problems

### Problem 1: Static Supply (where to put feeders)

**Given:** A settlement, N pawns, M candidate fertile hexes, a queen at a fixed location.
**Decide:** Which pawn goes to which hex.
**Objective:** Maximize food delivered to queen per turn.

This is basically the Hungarian assignment we already do, but with richer constraints
(sight cones, terrain, capacity limits). Good **starter problem** for Pyomo.

```
Sets:    F = feeders, C = candidate stations
Vars:    assign[f,c] ∈ {0,1}
Constr:  Σ_c assign[f,c] = 1              (each feeder → one station)
         Σ_f assign[f,c] ≤ station_cap    (don't overcrowd)
Obj:     maximize Σ_f,c assign[f,c] × delivery_rate[f,c]
```

### Problem 2: March Supply Chain (temporal)

**Given:** A march path with queen arrival times, candidate station hexes along it,
feeder pieces with positions/speeds.
**Decide:** Station assignments + food transfer schedule across T turns.
**Objective:** Maximize min(queen food at any turn) — i.e. keep her above starvation.

This is the big one. It's a multi-period LP with binary assignment + continuous food flows.

```
Sets:    P = all pieces, F ⊂ P = feeders, T = turns
Vars:    assign[f,c] ∈ {0,1}, food[p,t] ≥ 0, give[f,p,t] ≥ 0
Constr:  food balance (per piece per turn)
         capacity limits
         sight cone (give only if visible)
         queen position fixed to march_path[t]
Obj:     maximize min_t food[queen, t]
```

Size estimate: 6 pieces × 30 turns = 180 food vars + 6×6×30 = 1080 give vars
+ 6×10 assign vars ≈ 1300 variables. HiGHS eats this for breakfast.

### Problem 3: Mission Replan (mid-sim, fast)

**Given:** Current piece positions, food levels, remaining march path, possibly
a new threat hex (combat).
**Decide:** New assignments, possibly new target.
**Objective:** Same as #2 but from current state, or switch to "maximize
distance from threat" if combat_pressure is high.

This is Problem 2 re-solved with updated initial conditions. The model structure
is identical — only the parameters change. This is where Pyomo shines: you update
`m.food[p, 0] = current_food[p]` and re-solve.

---

## Architecture: How It Fits Together

```
┌──────────────────────────────────────────────────────┐
│  Mission                                              │
│    .target, .formation, .soldiers, .suppliers          │
│    .plan() → calls build_model() + solve()            │
│    .evaluate() → MissionStatus                        │
│    .replan() → rebuild model from current state        │
│    .as_callback() → SquadCallback for FoodSimulator    │
├──────────────────────────────────────────────────────┤
│  SupplyModel (Pyomo wrapper)                          │
│    .build(pieces, stations, path, tiers, turns)       │
│    .solve() → extracts assign[], food[], give[]       │
│    .to_orders() → {piece_id: InstructionList}         │
│    .to_df() → pd.DataFrame of optimal food schedule   │
│    .__ft__() → visual comparison: plan vs actual       │
├──────────────────────────────────────────────────────┤
│  FoodSimulator (existing)                             │
│    executes the plan, fires callbacks each turn        │
│    callbacks can trigger Mission.replan()              │
├──────────────────────────────────────────────────────┤
│  Overlays (existing pattern)                          │
│    MissionOverlay — target, path progress, grade       │
│    SupplyModelOverlay — optimal vs actual food curves  │
│    SquadFoodTrailOverlay — what actually happened      │
└──────────────────────────────────────────────────────┘
```

---

## The FastHTML-Style API Goal

Just like overlays are composable one-liners:

```python
TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63"),
    MissionOverlay(mission),
    PieceOverlay([squad]),
    terrain=terr, basins=basins, region=region
)
```

The optimization should feel the same:

```python
# Build + solve in one shot
model = SupplyModel.from_mission(mission)
model.solve()
show(model)  # __ft__ renders optimal food schedule chart

# Extract orders
orders = model.to_orders(grid)
for p in all_pieces:
    if p.id in orders:
        p.instructions = orders[p.id]

# Or even more concise:
mission.plan()  # internally builds model, solves, assigns instructions
sim.run(30)
show(mission)   # __ft__ renders status timeline + grade
```

---

## Key Design Decisions to Make

### 1. Visibility precomputation

The `give[f,p,t]` variable needs to know: can feeder f see piece p at turn t?
For a stationary feeder this is just the sight cone from its station.
For a moving queen, her position changes each turn.

**Approach:** Precompute a binary matrix `can_give[f, c, t]` before building the model.
For each candidate station c, for each turn t, check if `march_path[t]` is in
the sight cone from c. This is O(stations × turns × sight_hexes) — tiny.

### 2. How granular are "turns" in the model?

The sim has sub-turn slots (move_strength budget). The Pyomo model probably
works at **turn granularity** — one harvest + one give per turn per feeder.
This is a simplification but matches the pawn's typical H→G→H→G patrol.

### 3. Fixed queen path vs optimized queen path

Start with **fixed queen path** (from pathfind). The model only optimizes
feeder placement. Later, you could let the model suggest pauses:

```python
# Future: queen can choose to pause at fertile hexes
m.queen_pause = pyo.Var(m.T, domain=pyo.Binary)  # 1 = pause this turn
# ... adjust queen position schedule based on pauses
```

### 4. Model → InstructionList translation

The model outputs `assign[feeder, station]`. Translating to InstructionList:

```python
def to_orders(self, grid):
    orders = {}
    for f in self.feeders:
        for c in self.stations:
            if pyo.value(self.m.assign[f.id, c]) > 0.5:
                # Pathfind feeder → station
                path = f.pathfind(c, grid, self.elevations)
                facing = _facing_toward(grid, c, self.path_hex_for[c])
                rules = path.to_rules(grid, f.facing)
                rules += rotate_to(f.facing, facing)
                rules += [Instruction.HARVEST.value, Instruction.GIVE.value]
                orders[f.id] = InstructionList(rules, patrol=True)
    return orders
```

---

## Notebook Roadmap

### Cell group 1: Solver check
```python
import pyomo.environ as pyo
# Test HiGHS / GLPK / CBC availability
# Tiny 2-variable LP to confirm it works
```

### Cell group 2: Static supply model (Problem 1)
- Use PieceBoard.hilly test board
- Fixed queen, 5 pawns, 10 candidate stations
- Build model, solve, extract assignments
- Compare to Hungarian result — should match or beat

### Cell group 3: Temporal march model (Problem 2)
- Queen marches 20 hexes
- Model decides station placement + food schedule
- `show(model)` — chart of optimal food[queen, t]
- Convert to InstructionLists, run in FoodSimulator
- Compare model prediction vs sim actual (they should match closely)

### Cell group 4: Replan demo (Problem 3)
- Run sim 10 turns
- Inject a "punch" — kill a pawn, or block the path
- Rebuild model from current state
- Show before/after plans

### Cell group 5: Mission integration
- Wire model into Mission.plan()
- Mission.as_callback() triggers replan when status degrades
- Full 40-turn sim with adaptive supply chain
- MissionOverlay on the map

---

## Data Flow Cheat Sheet

```
Terrain + food_tiers + pieces
    │
    ▼
SupplyPlanner.plan_march(path)
    │  (candidate stations, timing, spacing)
    ▼
SupplyModel.build(pieces, stations, path, turns)
    │  (Pyomo model: sets, params, vars, constraints, objective)
    ▼
SupplyModel.solve()
    │  (HiGHS finds optimal assignment + food schedule)
    ▼
SupplyModel.to_orders(grid)
    │  ({piece_id: InstructionList})
    ▼
FoodSimulator.run()
    │  (executes orders, fires callbacks)
    ▼
Mission.evaluate() → MissionStatus
    │
    ├─ grade OK → continue
    └─ needs_replan → rebuild model from current state → new orders
```

---

## What We Keep vs What Changes

| Component | Status | Notes |
|-----------|--------|-------|
| FoodSimulator | ✅ keep | Execution engine, no changes needed |
| SupplyPlanner | 🔄 refactor | `.plan_march()` becomes input to Pyomo model |
| supply_line (Squad patch) | ❌ replace | Greedy heuristic → Pyomo optimal |
| Hungarian matching | ✅ keep for formations | Formation slots are a pure assignment problem |
| sync_paths | ✅ keep | Pyomo handles food timing, sync handles arrival timing |
| MissionStatus | ✅ keep | Evaluation is independent of planning method |
| Overlays | ✅ keep + add | Add SupplyModelOverlay for plan visualization |
| food_triage callback | 🔄 evolve | Becomes the "replan" trigger inside Mission |

---

## One More Thing: The AMPL Aesthetic

The real win isn't just "better solutions" — it's that the **problem statement
becomes the documentation**. When someone reads:

```python
m.food_bal = Constraint(m.P, m.T, rule=lambda m, p, t:
    m.food[p,t] == m.food[p,t-1] - m.diet[p] + m.harvest[p,t] + m.received[p,t])
```

They understand the food system. No need to trace through simulator loops.
The model IS the spec. The simulator validates it. The overlays visualize it.
That's the chocolate and peanut butter. 🍫🥜


In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc

import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 

from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets


from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds




from enum import Enum

In [ ]:
#| export
import pyomo.environ as pyo
import optuna

Based on your integration notes, you need:

```bash
!pip install pyomo highspy
```

- **`pyomo`** — the modeling library itself
- **`highspy`** — the Python bindings for HiGHS, which lets Pyomo use it via `SolverFactory('appsi_highs')`

You already have `scipy` (which bundles HiGHS internally), but Pyomo talks to HiGHS through `highspy` directly rather than through scipy's wrapper.

After installing, verify with a quick smoke test:

```python
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.x = pyo.Var(bounds=(0, 10))
m.obj = pyo.Objective(expr=m.x, sense=pyo.maximize)

solver = pyo.SolverFactory('appsi_highs')
result = solver.solve(m)
print(f"Status: {result.solver.termination_condition}, x = {pyo.value(m.x)}")
```

If that prints `Status: optimal, x = 10.0` you're good to go.

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

In [ ]:
showDemo = False
myStuff = GameParts()

In [ ]:
#| export
@patch
def __ft__(self: Squad, grid: HexGrid = None, list_pieces: bool = False):
    """Compact squad dashboard — header with icon + name, box plots by piece type."""
    alive = self.alive
    all_pieces = self.pieces

    # ── Header: icon (left) + text (center) + flag (right) ──
    icon_html = None
    flag_builder = None
    if self.flag is not None:
        animal = self.animal or self.flag.animal_name()
        icon_svg = self.flag.animal_svg(size='large', animal=animal,
                                         piece_id=f"squad_icon_{self.id}")
        r = int(22.5 * 3.5 * 1.3) + 5
        svg_size = r * 2 + 8
        icon_html = NotStr(
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{svg_size}" height="{svg_size}" '
            f'viewBox="0 0 {svg_size} {svg_size}">{icon_svg}</svg>'
        )
        flag_builder = self.flag.flag_banner(width=90, height=60, wavy=True, pole=True)

    squad_label = self.name or f"Squad #{self.id}"
    total_alive = len(alive)
    total_all = len(all_pieces)

    text_parts = [Strong(squad_label, style="font-size:1.2em")]
    if self.countryName is not None:
        text_parts.append(P(self.countryName, cls="text-sm opacity-70"))
    text_parts.append(P(f"🗡 {total_alive}/{total_all} alive", cls="text-sm"))

    header_parts = []
    if icon_html:     header_parts.append(Div(icon_html, cls="shrink-0"))
    header_parts.append(Div(*text_parts, cls="space-y-1 flex-1 text-center"))
    if flag_builder:  header_parts.append(Div(NotStr(flag_builder.xml()), cls="shrink-0"))

    header = Div(*header_parts, cls="flex items-center gap-4 mb-3")

    # ── Early out if no pieces — still show header ──
    if not all_pieces:
        return Div(header, P("No pieces assigned", cls="text-sm opacity-50 ml-2"),
                   id=f"squad-{self.id}", cls="space-y-2")

    # ── Group by piece type (ranked order) ──
    types_present = sorted(set(p.piece_type for p in all_pieces),
                           key=lambda t: RANK_ORDER.get(t, 99))
    type_labels = [t.name.title() for t in types_present]

    health_data, food_data, turns_data, distance_data = [], [], [], []
    alive_counts, total_counts = [], []

    centroid = None
    if grid is not None:
        locs = [p.location for p in alive
                if p.location is not None and p.location >= 0]
        if len(locs) >= 2:
            positions = [grid.index_to_hexposition(loc) for loc in locs]
            cq = sum(p.q for p in positions) / len(positions)
            cr = sum(p.r for p in positions) / len(positions)
            cs = sum(p.s for p in positions) / len(positions)
            centroid = (cq, cr, cs)

    for pt in types_present:
        members = [p for p in all_pieces if p.piece_type == pt]
        alive_m = [p for p in members if p.health > 0]
        alive_counts.append(len(alive_m))
        total_counts.append(len(members))
        health_data.append([p.health for p in members])
        food_data.append([p.food for p in members])
        turns_data.append([p.food / max(p.diet, 0.01) for p in members])

        if centroid is not None:
            dists = []
            for p in members:
                if p.location is not None and p.location >= 0:
                    pos = grid.index_to_hexposition(p.location)
                    d = (abs(pos.q - centroid[0])
                       + abs(pos.r - centroid[1])
                       + abs(pos.s - centroid[2])) / 2
                    dists.append(d)
                else:
                    dists.append(float('nan'))
            distance_data.append(dists)

    # ── Build figure ──
    has_dist = centroid is not None and distance_data
    n_panels = 4 if has_dist else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 3.0))
    if n_panels == 1: axes = [axes]

    panels = [
        ("Health",            health_data,   "#e74c3c"),
        ("Food",              food_data,     "#27ae60"),
        ("Food (turns left)", turns_data,    "#f39c12"),
    ]
    if has_dist:
        panels.append(("Distance from center", distance_data, "#3498db"))

    for ax, (title, data, color) in zip(axes, panels):
        if not data or all(len(d) == 0 for d in data):
            ax.set_visible(False)
            continue
        bp = ax.boxplot(data, tick_labels=type_labels, patch_artist=True,
                        medianprops=dict(color='#222', linewidth=1.5),
                        whiskerprops=dict(color='#888'),
                        capprops=dict(color='#888'))
        for patch in bp['boxes']:
            patch.set_facecolor(color); patch.set_alpha(0.55)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(axis='y', alpha=0.3, linewidth=0.5)
        ymax = ax.get_ylim()[1]
        for i, (a, t) in enumerate(zip(alive_counts, total_counts)):
            ax.text(i + 1, ymax * 1.02, f"{a}/{t}",
                    ha='center', va='bottom', fontsize=7, color='#555',
                    fontstyle='italic')

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    avg_hp = np.mean([p.health for p in alive]) if alive else 0
    min_hp = min((p.health for p in alive), default=0)
    avg_turns = np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0

    summary = P(
        f"❤ avg {avg_hp:.0f} (min {min_hp:.0f}) · "
        f"🌾 {avg_turns:.1f} avg food turns",
        cls="text-xs opacity-60 text-center mt-1"
    )

    parts = [header, NotStr(buf.getvalue()), summary]
    if list_pieces:
        parts.append(Hr())
        parts.append(PieceList(all_pieces))

    return Div(*parts, id=f"squad-{self.id}", cls="space-y-2")


In [ ]:
#| export
@patch
def food_gauge(piece:Piece, w=220, h=32, bar_h=14, bar_y=4):
    """Inline SVG food-turns-remaining gauge for a Piece card."""
    turns_left = piece.food / piece.diet if piece.diet > 0 else float('inf')
    max_turns = piece.food_capacity / piece.diet if piece.diet > 0 else 1
    reserve = FOOD_RESERVE_TURNS

    frac = min(turns_left / max_turns, 1.0) if max_turns > 0 else 1.0
    reserve_frac = min(reserve / max_turns, 1.0) if max_turns > 0 else 0

    # ── Styles ──
    canvas = SVGBuilder()
    canvas.width, canvas.height = w, h

    track = StyleCSS("fg_track", fill="#eee", stroke="none", stroke_width=0)

    if turns_left >= reserve:
        bar_color = StyleCSS.lerp_color('#e74c3c', '#27ae60', min(frac, 1.0))
    elif turns_left >= reserve * 0.5:
        bar_color = '#f39c12'
    else:
        bar_color = '#e74c3c'

    fill = StyleCSS("fg_fill", fill=bar_color, stroke="none", opacity=0.8)
    tick = StyleCSS("fg_tick", stroke="#e67e22", fill="none",
                    stroke_width=2, stroke_dasharray="2,1")
    lbl  = StyleCSS("fg_lbl", fill="#555", stroke="none", stroke_width=0)
    rlbl = StyleCSS("fg_rlbl", fill="#e67e22", stroke="none", stroke_width=0)

    for s in (track, fill, tick, lbl, rlbl):
        canvas.add_style(s)

    # ── Track + fill bar ──
    body  = f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" class="{track.name}"/>'
    body += f'<rect x="0" y="{bar_y}" width="{w * frac:.1f}" height="{bar_h}" rx="3" class="{fill.name}"/>'

    # ── Reserve tick ──
    body += MapPath([
        MapCord(w * reserve_frac, bar_y - 1),
        MapCord(w * reserve_frac, bar_y + bar_h + 1),
    ], tick).drawPolygon()

    # ── Labels ──
    txt_y = bar_y + bar_h + 14
    tl_str = f"{turns_left:.1f}" if turns_left < 1000 else "∞"
    body += (f'<text x="4" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" class="{lbl.name}">'
             f'🌾 {tl_str} turns left</text>')
    body += (f'<text x="{w - 4}" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" text-anchor="end" class="{rlbl.name}">'
             f'reserve={reserve}</text>')

    canvas.adjust("gauge", body)
    return NotStr(canvas.xml())


In [ ]:
#| export
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.flag, self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),
        DivCentered(self.food_gauge()),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )


In [ ]:
#| export
@patch
def place_squad_spiral(settlement: Settlement, squad, terrain, max_ring=10):
    """Place squad pieces spiral from center, low rank first. Provisions starting food."""
    grid = terrain.hexGrid
    center = settlement.location

    occupied = {center}
    for citizen in settlement.citizens:
        if citizen.location is not None:
            occupied.add(citizen.location)

    # Sort: pawns first (low rank), queens last (high rank)
    RANK_ORDER = {
        PieceType.PAWN: 0, PieceType.ROOK: 1, PieceType.BISHOP: 2,
        PieceType.KNIGHT: 3, PieceType.QUEEN: 4, PieceType.KING: 5,
    }
    sorted_pieces = sorted(squad.pieces,
                           key=lambda p: RANK_ORDER.get(p.piece_type, 99))

    spiral = HexPosition.origin().spiral(max_ring)

    placed = 0
    for hp in spiral[1:]:
        if placed >= len(sorted_pieces):
            break

        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0: continue
        if terrain.elevations[idx] <= 0: continue
        if idx in occupied: continue

        piece = sorted_pieces[placed]
        piece.location = idx
        piece.instructions = InstructionList([Instruction.HARVEST.value])

        # Provision: fill to capacity based on local food tier
        piece.food = piece.food_capacity

        occupied.add(idx)
        placed += 1

    return placed

In [ ]:

TerrainDisplay(
    CreamOverlay(stylized=True),
    SquadSymbolOverlay([myStuff.queenGuard]),
    #SettlementOverlay(scale=1),
    #KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    SquadVisionOverlay([myStuff.queenGuard]),
    terrain=myStuff.terr,
    region=myStuff.region(), padding=2,
    board=myStuff.board,
    debug = not showDemo
)



## FoodYield

In [ ]:
#| export


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
#| export
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
# Make sure soil exists

soil = SoilSystem.from_plates(myStuff.terr, [])

fy = FoodYield(myStuff.terr, myStuff.basin)
fy.compute()
#show(fy)
myStuff.fy = fy
fy.summary()

In [ ]:
show(fy)

In [ ]:
#| export


def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


## FoodOverlay

In [ ]:
#| export
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:

myStuff.clearTerr()
myStuff.terr.repalette()
TerrainDisplay(
    TerrainOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(max_width=6),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    debug = not showDemo
)


In [ ]:
#!cat ../../HexMagic/game/piece.py

## Food Profile

Great balancing question. 

## Food Economy — Piece Balancing

### Design Philosophy

Each piece type has a **logistical role** in the food economy:

| Role | Piece | Design Intent |
|------|-------|---------------|
| **Gatherer** | Pawn | High harvest strength, tiny personal reserves. Produces surplus food for others. |
| **Food Bank** | Bishop | Massive capacity (178.7!), moderate harvest. Stores and redistributes surplus across legs. |
| **Consumer** | Queen | Extremely high diet (27.9/turn), large capacity but negligible harvest. *Cannot self-sustain on any terrain.* |
| **Heavy Foot** | Rook | *(not yet tuned)* Moderate diet, self-sustaining on decent terrain. |
| **Fast Raider** | Knight | *(not yet tuned)* Higher diet, self-sustaining, speed advantage. |
| **Precious** | King | Low harvest, moderate reserves. Needs at least one pawn escort. |

The core design tension: **the queen is always in caloric deficit**. On the best terrain (tier 7),
her net is still −24.5/turn. This makes escort composition a meaningful strategic decision rather
than a cosmetic one.

### How We Got Here

**Step 1 — Supply Line Sim.** We built a `FoodSimulator` that runs turn-by-turn with instructions
(FORWARD, HARVEST, GIVE, PAUSE). Early experiments showed a lone queen starving by turn 5–6, and
even with pawns in a supply line she only survived to turn 12–15 due to the speed mismatch
(queen speed 5 vs pawn speed 2 at the time).

**Step 2 — Leapfrog Planner.** We replaced the per-turn sim with an analytical `march_range` +
`leapfrog_plan` approach: trace a route, find the farthest waypoint the whole squad can reach
with food ≥ 0, rest and recover at that waypoint, repeat. This gave us a fast
`run_scenario(profiles, n_queens, n_bishops, n_pawns) → range%` function.

**Step 3 — Recovery Model.** At each waypoint we run `squad_recover(squad, food_tier, recovery_turns)`:
1. Each piece harvests and eats independently
2. Surplus overflows to bishops first (they're the bank), then others
3. Deficit pieces draw from bishops first, then richest donors

Before each leg, `pre_march_distribute` has the bishop top up the queen from its reserves.

**Step 4 — Scenario Table.** Inspired by
[Paul Tozour's PvP balancing series](https://intelligenceengine.blogspot.com/2013/07/decision-modeling-and-optimization-in_28.html),
we defined a "victory table" of squad compositions, each with a target range%:

```
(1Q, 0B, 0P) →   5%   Queen alone: nearly dead
(1Q, 1B, 0P) →  20%   Bishop helps but can't harvest enough
(1Q, 0B, 2P) →  65%   Pawns harvest, no bank — capped
(1Q, 0B, 3P) →  75%   3rd pawn wasted without bank
(1Q, 1B, 2P) →  90%   Sweet spot — bishop + pawns
(1Q, 1B, 4P) → 100%   Easy
(4Q, 1B, 4P) →  35%   4 queens overwhelm 1 bishop
(4Q, 2B, 8P) →  90%   Proper army
```

**Step 5 — Optuna Optimization.** Since `leapfrog_plan` is a black-box sim (pathfinding,
terrain, visibility cones), we can't linearize it into Pyomo. Instead we used Optuna with
`MedianPruner` to search over 9 parameters (diet, capacity, harvest for queen/bishop/pawn)
across 300 trials. The objective minimized weighted absolute error from the target table,
plus penalty terms enforcing role identity constraints (queen diet >> bishop diet, pawn harvest
>> bishop harvest, bishop capacity >> queen capacity, etc.).

**Result:** Trial 108 scored best (error 122.6). Most scenarios landed within 3–10% of target.
The main remaining gap is `(1Q, 0B, 3P)` hitting 100% instead of 75% — meaning 3 pawns alone
can fully sustain a queen without a bishop on this map. This could be addressed by lowering
`pawn_harvest` below 2.0 or making the route longer/harder.

### How to Rerun for New Pieces

When tuning rook and knight (or retuning after mechanic changes):

1. **Pre-build the shared map** (expensive, do once):
```python
_map = GameParts(); _map.profile()
_fy = FoodYield(_map.terr, _map.basin); _fy.compute()
_food_tiers = _fy.tiers
_grid, _elevs = _map.grid, _map.terr.elevations
_countries = _map.terr.fields.get('country')
_home = _map.home
```

2. **Add new piece types to `run_scenario`** — extend the
   `(PieceType, count)` list and the `profiles` dict.

3. **Design new scenarios** that test the new piece's role. For example:
```python
# Rook scenarios — should self-sustain but not replace bishop
(0, 0, 0, 1, 0, 0,  60%),  # 1 rook alone: decent range
(1, 0, 0, 1, 0, 2,  85%),  # queen + rook + 2 pawns: rook helps but isn't a bank

# Knight scenarios — fast but hungry
(0, 0, 0, 0, 1, 0,  40%),  # 1 knight alone: faster but burns food
(1, 0, 0, 0, 1, 2,  80%),  # queen + knight + 2 pawns: knight scouts ahead
```

4. **Expand `balance_objective`** — add the new `suggest_float` params with
   role-appropriate bounds and identity constraints.

5. **Run Optuna** — 300 trials takes ~3.5 min. Use `load_if_exists=True` to
   resume from a previous study:
```python
study = optuna.create_study(
    study_name="piece_balance_v6",
    storage="sqlite:///balance.db",
    direction="minimize",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=3),
)
study.optimize(balance_objective, n_trials=300, show_progress_bar=True)
```

6. **Lock existing values** — if you only want to tune rook/knight without
   disturbing the existing balance, hardcode queen/bishop/pawn values and only
   `suggest_float` the new params.

### Key Files

- `FoodProfile` — the design values (this file)
- `squad_recover` / `pre_march_distribute` — the food economy logic
- `run_scenario` — the black-box sim wrapper
- `balance_objective` — the Optuna objective function
- `balance.db` — SQLite study history (viewable via `optuna-dashboard`)


In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

#| export
class FoodProfile:
    """Tuned food stats per piece type.

    Design intent & roles:
      PAWN   = gatherer    — harvests for others, tiny personal reserves
      BISHOP = food bank   — massive capacity, moderate harvest, bankrolls the queen
      QUEEN  = consumer    — high diet, barely self-sufficient, needs escort
      ROOK   = heavy foot  — moderate diet, self-sustaining on good terrain
      KNIGHT = fast raider — higher diet, self-sustaining, speed advantage
      KING   = precious    — low harvest, moderate reserves, must be protected

    Tuned via Optuna (300 trials) against scenario table:
      Queen alone          →  ~5% range
      Queen + Bishop       → ~22% range
      Queen + 2 Pawns      → ~63% range
      Queen + 3 Pawns      → ~75% range  (bank-less ceiling)
      Queen + Bishop + 2P  → ~90% range  (sweet spot)
      Queen + Bishop + 4P  → 100% range
    """

    PROFILES = {
        #                        diet    cap   harv  spd
        PieceType.PAWN:   (      2.7,    8.7,  2.06,  2),
        PieceType.BISHOP: (      1.9,  178.7,  0.84,  3),
        PieceType.ROOK:   (      3.5,   12.0,  1.8,   3),
        PieceType.KNIGHT: (      4.0,   10.0,  1.5,   4),
        PieceType.QUEEN:  (     27.9,   59.0,  0.48,  3),
        PieceType.KING:   (      5.0,   14.0,  0.5,   2),
    }

    @classmethod
    def apply(cls, piece):
        """Apply profile stats to a piece in place. Returns the piece."""
        diet, cap, harv, spd = cls.PROFILES.get(
            piece.piece_type, (1.0, 8.0, 2.0, 3))
        piece.diet             = diet
        piece.food_capacity    = cap
        piece.harvest_strength = harv
        piece.move_strength    = spd
        piece.food             = cap
        return piece

    @classmethod
    def summary(cls):
        """Print a human-readable table of all profiles."""
        print(f"{'Type':8s} {'Diet':>6s} {'Cap':>7s} {'Harv':>5s} "
              f"{'Spd':>4s} {'Turns':>6s} {'Net@T5':>8s} {'Net@T7':>8s}")
        print("─" * 60)
        for pt, (d, c, h, s) in cls.PROFILES.items():
            turns  = c / d if d > 0 else float('inf')
            net_t5 = h * 5 - d
            net_t7 = h * 7 - d
            print(f"{pt.name:8s} {d:6.1f} {c:7.1f} {h:5.2f} "
                  f"{s:4d} {turns:6.1f} {net_t5:+8.2f} {net_t7:+8.2f}")
        print()
        print("Net@T5/T7 = harvest surplus per turn on tier-5/7 tile.")
        print("Positive = self-sustaining. Negative = needs support.")

FoodProfile.summary()


@patch
def profile(self:GameParts):
    for person in self.home.citizens:
        FoodProfile.apply(person)

myStuff.profile()

## Mecanics

In [ ]:
#| export


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)



def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)


In [ ]:
#| export
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         battlefield: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone.
    
    battlefield is ALL pieces (across all squads) — give targets
    any allied piece in sight, not just squad-mates.
    """
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            grid, elevations, battlefield,
            facing_only=True, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount


## TroopPath

In [ ]:
#| export
@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier. Returns TroopPath (falsy if none)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)



In [ ]:
#| export
@patch
def food_audit(self: Squad, food_tiers: np.ndarray) -> pd.DataFrame:
    """Static food analysis — one row per alive piece."""
    rows = []
    for p in self.alive:
        tier = int(food_tiers[p.location]) if (p.location is not None and p.location >= 0) else 0
        gross = p.harvest_strength * tier
        net = gross - p.diet
        rows.append({
            'piece': p,
            'name': p.name,
            'type': p.piece_type.name,
            'diet': p.diet,
            'harvest_str': p.harvest_strength,
            'tier': tier,
            'gross': gross,
            'net': net,
            'food': p.food,
            'reserve_turns': p.food / p.diet if p.diet > 0 else float('inf'),
            'location': p.location,
        })
    df = pd.DataFrame(rows)
    df.attrs['demand'] = df.diet.sum()
    df.attrs['supply'] = df.loc[df.net > 0, 'net'].sum()
    df.attrs['gap'] = df.attrs['demand'] - df.attrs['supply'] - df.loc[df.net <= 0, 'gross'].sum()
    return df


In [ ]:
#| export
def PieceArrowOverlay(df: pd.DataFrame, column: str = 'net',
                      color: str = '#555555',
                      max_width: float = 4.0, min_width: float = 1.0,
                      opacity: float = 0.6,
                      head_size: float = 8.0, **kw) -> OverlaySpec:
    """Draw arrows from sources (column > 0) to nearest sinks (column < 0).
    
    Each sink connects to its nearest source. Arrow width ∝ sink demand.
    """
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)

        sources = df[df[column] > 0]
        sinks = df[df[column] < 0]

        if sources.empty or sinks.empty:
            return ''

        # Marker definition for arrowhead
        marker_id = "food_arrow_head"
        marker_svg = (
            f'<marker id="{marker_id}" viewBox="0 0 10 10" '
            f'refX="9" refY="5" markerWidth="{head_size:.0f}" '
            f'markerHeight="{head_size:.0f}" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}" '
            f'opacity="{opacity:.2f}"/></marker>')
        ctx.builder.add_definition(SVGDef("", marker_id, marker_svg, raw=True))

        # Pre-compute source centers (mapped for zoom)
        src_locs = sources['location'].values.astype(int)
        src_centers = []
        for loc in src_locs:
            render_idx = _map_point(loc, c2f) if c2f is not None else loc
            if 0 <= render_idx < len(grid.hexes):
                c = grid.hexes[render_idx].center
                src_centers.append((render_idx, c.x, c.y))

        if not src_centers:
            return ''

        max_demand = max(abs(sinks[column].min()), 0.01)

        parts = []
        for _, row in sinks.iterrows():
            sink_loc = int(row['location'])
            render_idx = _map_point(sink_loc, c2f) if c2f is not None else sink_loc
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue
            sc = grid.hexes[render_idx].center

            # Find nearest source by pixel distance
            best_dist, best_src = float('inf'), None
            for s_loc, sx, sy in src_centers:
                d = math.hypot(sx - sc.x, sy - sc.y)
                if d < best_dist:
                    best_dist, best_src = d, (sx, sy)

            if best_src is None:
                continue

            # Width proportional to demand magnitude
            t = min(abs(row[column]) / max_demand, 1.0)
            w = min_width + (max_width - min_width) * t

            sx, sy = best_src
            parts.append(
                f'<line x1="{sx:.1f}" y1="{sy:.1f}" '
                f'x2="{sc.x:.1f}" y2="{sc.y:.1f}" '
                f'stroke="{color}" stroke-width="{w:.1f}" '
                f'opacity="{opacity:.2f}" '
                f'marker-end="url(#{marker_id})"/>')

        return '\n'.join(parts)

    return OverlaySpec("piece_arrows", render, priority=73)


In [ ]:
#| export
def PieceDataOverlay(df: pd.DataFrame, column: str = 'net',
                     pos_color: str = '#27ae60', neg_color: str = '#e74c3c',
                     zero_color: str = '#eeeeee',
                     opacity: float = 0.45, **kw) -> OverlaySpec:
    """Color-fill each piece's hex by a DataFrame column value."""
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)
        vals = df[column].values.astype(float)
        locs = df['location'].values.astype(int)

        bound = max(abs(vals.min()), abs(vals.max()), 0.01)

        parts = []
        for val, loc in zip(vals, locs):
            # Map coarse → fine for zoomed views
            render_idx = _map_point(loc, c2f) if c2f is not None else loc
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            t = np.clip(val / bound, -1, 1)
            if t >= 0:
                color = StyleCSS.lerp_color(zero_color, pos_color, t)
            else:
                color = StyleCSS.lerp_color(zero_color, neg_color, -t)

            h = grid.hexes[render_idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" '
                f'opacity="{opacity:.2f}" stroke="{color}" '
                f'stroke-width="1.5" stroke-opacity="0.6"/>')
        return '\n'.join(parts)

    return OverlaySpec("piece_data", render, priority=72)


In [ ]:
myStuff.clearTerr()
myStuff.terr.repalette()
myStuff.grid.adjustRadius(35)


In [ ]:
showDemo = False

In [ ]:
??TerrainOverlay

In [ ]:
myStuff.food_tiers = myStuff.fy.tiers
audit = myStuff.queenGuard.food_audit(myStuff.food_tiers)


TerrainDisplay(
    TerrainOverlay(stylized=True),
    FoodOverlay(color="#cf35b0ff", n_tiers=6),
    SettlementOverlay(),
    KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    PieceDataOverlay(audit, 'net'),
    PieceArrowOverlay(audit, 'net'),
    SquadSymbolOverlay([myStuff.queenGuard]),
    SquadSymbolOverlay([myStuff.queenGuard], sim_grid=myStuff.grid),
    #SquadVisionOverlay([myStuff.queenGuard], sim_grid=myStuff.grid),
    #SquadDietOverlay([queenGuard], food_tiers, sim_grid=myGrid),
    RiverOverlay(max_width=6),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    region=myStuff.region(),
    board = myStuff.board,
    debug = not showDemo
)
        



In [ ]:
#| export
@patch
def supply_line(feeders: Squad, path: list[int], 
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(HexPosition.origin().field_of_view( facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## useful things

In [ ]:
#| export
def pre_march_distribute(squad):
    """Before each leg, bishop tops up hungrier squad members.
    
    Donors keep at least 2 turns of food for themselves.
    Neediest (by food/capacity ratio) are filled first.
    """
    for p in sorted(squad.alive, key=lambda x: x.food / x.food_capacity):
        if p.food >= p.food_capacity:
            continue
        need = p.food_capacity - p.food
        # Bishops donate first, then anyone else richest-first
        donors = sorted(
            squad.alive,
            key=lambda d: (d.piece_type != PieceType.BISHOP, -d.food)
        )
        for donor in donors:
            if donor.id == p.id:
                continue
            reserve = donor.diet * 2          # keep 2 turns for self
            give    = min(need, max(0, donor.food - reserve))
            if give <= 0:
                continue
            p.food    += give
            donor.food -= give
            need       -= give
            if need <= 0:
                break

In [ ]:
#| export
def squad_recover(squad, food_tier, recovery_turns=3):
    """Post-march harvest + surplus sharing.
    
    Phase 1 — each piece harvests and eats independently.
    Phase 2 — overflow goes to bishops (food bank), then others.
    Phase 3 — deficit pieces draw from bishops first, then richest.
    
    Returns (total_surplus, total_deficit) after all transfers.
    """
    surpluses = {}   # id → excess food that didn't fit
    deficits  = {}   # id → food still owed

    # Phase 1
    for p in squad.alive:
        production = p.harvest_strength * food_tier * recovery_turns
        consumed   = p.diet             *             recovery_turns
        new_food   = p.food + production - consumed

        if new_food > p.food_capacity:
            surpluses[p.id] = new_food - p.food_capacity
            p.food = p.food_capacity
        elif new_food < 0:
            deficits[p.id] = -new_food
            p.food = 0
        else:
            p.food = new_food

    total_surplus = sum(surpluses.values())
    total_deficit = sum(deficits.values())

    # Phase 2 — pour surplus into bishops first
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    others  = [p for p in squad.alive if p.piece_type != PieceType.BISHOP]

    for bank in sorted(bishops, key=lambda p: p.food):
        if total_surplus <= 0:
            break
        space         = bank.food_capacity - bank.food
        give          = min(space, total_surplus)
        bank.food    += give
        total_surplus -= give

    # Any remaining surplus spills into other pieces (least full first)
    for p in sorted(others, key=lambda p: p.food / max(p.food_capacity, 1)):
        if total_surplus <= 0:
            break
        space         = p.food_capacity - p.food
        give          = min(space, total_surplus)
        p.food       += give
        total_surplus -= give

    # Phase 3 — deficit pieces draw from bishops, then richest
    if total_deficit > 0:
        donors = sorted(
            squad.alive,
            key=lambda d: (d.piece_type != PieceType.BISHOP, -d.food)
        )
        for p in squad.alive:
            if p.id not in deficits:
                continue
            need = deficits[p.id]
            for donor in donors:
                if donor.id == p.id:
                    continue
                give       = min(need, max(0, donor.food - donor.diet))
                p.food    += give
                donor.food -= give
                need       -= give
                if need <= 0:
                    break

    return total_surplus, total_deficit


## The simulation

Good instinct, but the pinning isn't the culprit here — it's a **food filter** issue in `march_range`. Here's what's happening:

**The leapfrog planner fails because of `min_food_tier=2`**. Every candidate waypoint must have food tier ≥ 2 within 1 ring. If the settlement spawns in a food desert (and ~55% of your land hexes are tier 0–1), there may be **no qualifying waypoints** within the squad's travel range, so `march_range` scans all the way back to waypoint 0 and gives up.

**The food balance is also razor-thin**. Look at the starting `feed_cover`:

```
Production: 40.8 vs Diet: 41.4  ← short by 0.6!
```

Each pawn gets `(speed-1) × harvest × tier = 1 × 2.06 × tier` per feed cycle. With 5 pawns at tier 4: `5 × 2.06 × 4 = 41.2` — still barely under 41.4. You need tier **5+** for 1Q+5P to break even.

**That said, pawn-blocking IS a real problem in the simulator** (just not in the planner). The queen moves first in `_sort_move` priority order, tries `_try_forward`, finds a pawn occupying the forward hex, and gets `BLOCKED` — burning her action. The pawns move later but it's too late.

Three fixes to consider:

1. **Lower `min_food_tier` to 1** (or 0) so the planner doesn't filter out marginal terrain
2. **Add a bishop** — that massive capacity (178.7) bank is exactly designed for this; the queen needs a bishop escort
3. **In the simulator**, allow allies to swap hexes or skip occupied-by-ally checks (so the queen can push through friendly pawns)

In [ ]:
#| export
from typing import Callable

@dataclass
class TurnReport:
    """End-of-turn context passed to squad callbacks."""
    turn: int
    squad: Squad
    snapshot: dict
    food_audit: dict
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    countries: np.ndarray | None
    elevation_mult: float
    sim_df: pd.DataFrame

SquadCallback = Callable[[TurnReport], None]


In [ ]:
#| export
# Add to SimEventType
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    BUMPED    = "bumped"      # ← new
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"


_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND,
              Instruction.HARVEST, Instruction.GIVE, Instruction.SETTLE}

@patch
def _eat_phase(self: FoodSimulator, acted_ids: set = None):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue

        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
#| export
@dataclass
class FoodSimulator:
    """Turn-based simulator with per-slot interleaved execution."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    squads: list = field(default_factory=list)
    callbacks: dict = field(default_factory=dict)  # squad.id → SquadCallback

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    def __post_init__(self):
        self._squad_map = {s.id: s for s in self.squads}

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             slot=None, src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn': self.turn, 'slot': slot, 'event': etype.value,
            'piece_id': piece.id, 'piece_name': piece.name,
            'src_hex': src_hex if src_hex is not None else loc,
            'dst_hex': dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id, 'amount': amount,
        })

    def _sort_move(self, pieces):
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99), p.birth_year, -p.health, p.id))

    # ── actions (unchanged) ──
    def _do_harvest(self, piece, slot=None):
        loc = piece.location
        if loc is None or loc < 0: return
        tier = int(self.food_tiers[loc])
        gained = piece.harvest_strength * tier
        old = piece.food
        piece.food = min(piece.food + gained, piece.food_capacity)
        self._log(piece, SimEventType.HARVESTED, slot=slot, amount=piece.food - old)

    def _do_give(self, piece, slot=None):
        reserve = FOOD_RESERVE_TURNS * piece.diet
        surplus = max(0.0, piece.food - reserve)
        if surplus <= 0: return
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))
        receivers = sorted(
            [p for p in self.alive if p is not piece
             and p.owner_id == piece.owner_id
             and p.food < p.food_capacity
             and p.location in visible],
            key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))
        for recv in receivers:
            if surplus <= 0: break
            amount = min(surplus, recv.food_capacity - recv.food)
            recv.food += amount; piece.food -= amount; surplus -= amount
            self._log(piece, SimEventType.GAVE, slot=slot,
                      dst_piece_id=recv.id, amount=amount)

   

    def _tick_one(self, piece, budget, occupied, slot=None):
        il = piece.instructions
        if not il.rules: return None, 0.0
        while True:
            if il.cursor >= len(il.rules):
                if il.patrol: il.cursor = 0
                else: return None, 0.0
            instr = Instruction(il.rules[il.cursor])
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                self._log(piece, SimEventType.ROTATED, slot=slot)
                il.cursor += 1; continue
            if instr in _COSTS_ONE:
                if budget < 1.0: return None, 0.0
                il.cursor += 1
                if instr == Instruction.HARVEST: self._do_harvest(piece, slot)
                elif instr == Instruction.GIVE: self._do_give(piece, slot)
                return instr, 1.0
            if budget < 1.0: return None, 0.0
            cost = self._try_forward(piece, occupied, slot)
            if cost > 0:
                if budget >= cost: il.cursor += 1; return Instruction.FORWARD, cost
                else: return None, 0.0
            return None, 0.0

    def _eat_phase(self, acted_ids: set = None):
        for piece in list(self.pieces):
            if piece.health <= 0: continue
            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)
            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)
            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive), 'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0,
            'piece_states': {
                p.id: {'location': p.location, 'food': p.food,
                       'food_capacity': p.food_capacity, 'health': p.health}
                for p in self.pieces},
        })

    def _fire_callbacks(self):
        if not self.callbacks: return
        snap = self.snapshots[-1] if self.snapshots else {}
        for squad_id, cb in self.callbacks.items():
            squad = self._squad_map.get(squad_id)
            if squad is None or not squad.alive: continue
            report = TurnReport(
                turn=self.turn, squad=squad, snapshot=snap,
                food_audit=squad.food_audit(self.food_tiers),
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, countries=self.countries,
                elevation_mult=self.elevation_mult, sim_df=self.df,
            )
            cb(report)

    def step(self):
        alive = self.alive
        if not alive: return
        max_slots = max(p.move_strength for p in alive)
        budgets = {p.id: float(p.move_strength) for p in alive}
        acted = set()
        occupied = self.occupied
        order = self._sort_move(alive)
        for slot in range(max_slots):
            for piece in order:
                if piece.health <= 0: continue
                if budgets[piece.id] <= 0: continue
                instr, cost = self._tick_one(piece, budgets[piece.id], occupied, slot=slot)
                if instr is not None:
                    budgets[piece.id] -= cost; acted.add(piece.id)
        self._eat_phase()
        self._snapshot()
        self._fire_callbacks()
        self.turn += 1

    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive: self._snapshot(); break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")


In [ ]:
#| export
@patch
def _try_bump(self: FoodSimulator, mover, blocker, occupied, slot=None):
    """Swap mover (higher rank) with blocker (lower rank).
    
    Prepends recovery instructions to blocker:
      rotate → face old hex, FORWARD, rotate → restore original facing.
    Returns movement cost for the mover, or None if bump isn't allowed.
    """
    # Must be allies
    if mover.owner_id != blocker.owner_id:
        return None
    # Strictly higher rank
    if RANK_ORDER.get(mover.piece_type, 99) <= RANK_ORDER.get(blocker.piece_type, 99):
        return None
    
    old_mover_loc = mover.location
    old_blocker_loc = blocker.location
    old_blocker_facing = blocker.facing
    
    # ── Swap positions ──
    occupied.discard(old_mover_loc)
    occupied.discard(old_blocker_loc)
    mover.location = old_blocker_loc
    blocker.location = old_mover_loc
    occupied.add(old_blocker_loc)
    occupied.add(old_mover_loc)
    
    cost = _move_cost(self.elevations, old_mover_loc, old_blocker_loc)
    
    # ── Build recovery instructions for the bumped piece ──
    # Step 1: rotate to face old position (where it wants to return)
    target_facing = _facing_toward(self.grid, old_mover_loc, old_blocker_loc)
    
    diff = (target_facing - old_blocker_facing) % 6
    if diff <= 3:
        rot_to = [Instruction.ROT_R.value] * diff
    else:
        rot_to = [Instruction.ROT_L.value] * (6 - diff)
    
    # Step 2: FORWARD (will likely be blocked this turn, succeeds next)
    
    # Step 3: rotate back to original facing
    diff_back = (old_blocker_facing - target_facing) % 6
    if diff_back <= 3:
        rot_back = [Instruction.ROT_R.value] * diff_back
    else:
        rot_back = [Instruction.ROT_L.value] * (6 - diff_back)
    
    recovery = rot_to + [Instruction.FORWARD.value] + rot_back
    
    # Insert at current cursor (so recovery executes before existing plan)
    il = blocker.instructions
    il.rules = il.rules[:il.cursor] + recovery + il.rules[il.cursor:]
    # cursor stays — recovery is now what's next
    
    # ── Log ──
    self._log(mover, SimEventType.BUMPED, slot=slot,
              src_hex=old_mover_loc, dst_hex=old_blocker_loc,
              dst_piece_id=blocker.id)
    
    return cost


In [ ]:
#| export
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None):
    auto_rots = 0
    saved_fac = piece.facing
    while auto_rots <= 5:
        d = HexPosition.directions()[piece.facing % 6]
        nbr = self.grid.hexposition_to_index(d, piece.location)
        passable = (0 <= nbr < len(self.elevations)
                    and nbr not in self.grid.invalidRegion
                    and self.elevations[nbr] >= 1)
        if self.countries is not None and passable:
            passable = self.countries[nbr] >= 0
        if not passable:
            piece.facing = (piece.facing - 1) % 6; auto_rots += 1; continue
        if nbr in occupied and nbr != piece.location:
            blocker = next((p for p in self.alive if p.location == nbr), None)
            if blocker:
                bump_cost = self._try_bump(piece, blocker, occupied, slot)
                if bump_cost is not None:
                    return bump_cost
            self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
            return 1.0
        cost = _move_cost(self.elevations, piece.location, nbr)
        src = piece.location
        occupied.discard(src); piece.location = nbr; occupied.add(nbr)
        self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
        return cost
    piece.facing = saved_fac
    self._log(piece, SimEventType.BLOCKED, slot=slot)
    return 0.0

### Callbacks

In [ ]:
#| export
def food_triage(report: TurnReport):
    """Prepend harvest+give when squad is hungry."""
    avg_left = report.snapshot.get('avg_turns_left', 999)
    if avg_left >= FOOD_RESERVE_TURNS:
        return
    
    alive = [p for p in report.squad.pieces if p.health > 0]
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0:
            continue
        
        prepend = []
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier >= 1 and piece.food < piece.food_capacity:
            prepend.append(Instruction.HARVEST.value)
        
        reserve = FOOD_RESERVE_TURNS * piece.diet
        if piece.food > reserve:
            visible = piece.hexes_in_sight(
                report.grid, report.elevations,
                elevation_mult=report.elevation_mult,
                facing_only=True)
            hungry = any(
                p for p in alive
                if p is not piece and p.location in visible
                and p.food < p.food_capacity
            )
            if hungry:
                prepend.append(Instruction.GIVE.value)
        
        if prepend:
            piece.instructions.rules = prepend + piece.instructions.rules


In [ ]:
#| export
def compose_callbacks(*fns: SquadCallback) -> SquadCallback:
    """Chain multiple callbacks — each sees mutations from the previous."""
    def combined(report: TurnReport):
        for fn in fns:
            fn(report)
    return combined


### Analysis

In [ ]:
#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts: alive count, health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    df = pd.DataFrame(self.snapshots)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    # Alive
    axes[0].plot(df['turn'], df['alive'], 'o-', color='#2ecc71', lw=2, ms=4)
    axes[0].axhline(df['total'].iloc[0], color='#bbb', ls='--', lw=1)
    axes[0].set_title('Alive', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)
    axes[0].fill_between(df['turn'], df['alive'], alpha=0.15, color='#2ecc71')

    # Health
    axes[1].plot(df['turn'], df['avg_health'], 'o-', color='#e74c3c', lw=2, ms=4, label='avg')
    axes[1].plot(df['turn'], df['min_health'], 's--', color='#c0392b', lw=1, ms=3, alpha=0.6, label='min')
    axes[1].set_title('Health', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

    # Food turns remaining
    axes[2].plot(df['turn'], df['avg_turns_left'], 'o-', color='#f39c12', lw=2, ms=4)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left)', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].legend(fontsize=8); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")

So I want to add a bump to our world. a piece of a higher rank can swap places with a piece of a lower rank. the piece with the lower rank will have actions in the front of its queue where it will rotate to point where it was. one forward and one rotate back. when it their turn they will rotate, see that they can't move, but be set up for the next round.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great mechanic! The queen shoves a pawn aside, the pawn recovers next round. This solves the friendly-blocking problem elegantly.

Here's how I'd wire it into `_try_forward`. The changes are:

1. **New event type** — `BUMPED`
2. **Rank check** — only same-owner, strictly higher rank
3. **Swap positions** — both pieces move in the occupied set
4. **Prepend recovery instructions** — rotate toward old hex, FORWARD, rotate back

```python
# Add to SimEventType
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    BUMPED    = "bumped"      # ← new
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
```

And the bump logic itself — a helper method on `FoodSimulator`:

```python
@patch
def _try_bump(self: FoodSimulator, mover, blocker, occupied, slot=None):
    """Swap mover (higher rank) with blocker (lower rank).
    
    Prepends recovery instructions to blocker:
      rotate → face old hex, FORWARD, rotate → restore original facing.
    Returns movement cost for the mover, or None if bump isn't allowed.
    """
    # Must be allies
    if mover.owner_id != blocker.owner_id:
        return None
    # Strictly higher rank
    if RANK_ORDER.get(mover.piece_type, 99) <= RANK_ORDER.get(blocker.piece_type, 99):
        return None
    
    old_mover_loc = mover.location
    old_blocker_loc = blocker.location
    old_blocker_facing = blocker.facing
    
    # ── Swap positions ──
    occupied.discard(old_mover_loc)
    occupied.discard(old_blocker_loc)
    mover.location = old_blocker_loc
    blocker.location = old_mover_loc
    occupied.add(old_blocker_loc)
    occupied.add(old_mover_loc)
    
    cost = _move_cost(self.elevations, old_mover_loc, old_blocker_loc)
    
    # ── Build recovery instructions for the bumped piece ──
    # Step 1: rotate to face old position (where it wants to return)
    target_facing = _facing_toward(self.grid, old_mover_loc, old_blocker_loc)
    
    diff = (target_facing - old_blocker_facing) % 6
    if diff <= 3:
        rot_to = [Instruction.ROT_R.value] * diff
    else:
        rot_to = [Instruction.ROT_L.value] * (6 - diff)
    
    # Step 2: FORWARD (will likely be blocked this turn, succeeds next)
    
    # Step 3: rotate back to original facing
    diff_back = (old_blocker_facing - target_facing) % 6
    if diff_back <= 3:
        rot_back = [Instruction.ROT_R.value] * diff_back
    else:
        rot_back = [Instruction.ROT_L.value] * (6 - diff_back)
    
    recovery = rot_to + [Instruction.FORWARD.value] + rot_back
    
    # Insert at current cursor (so recovery executes before existing plan)
    il = blocker.instructions
    il.rules = il.rules[:il.cursor] + recovery + il.rules[il.cursor:]
    # cursor stays — recovery is now what's next
    
    # ── Log ──
    self._log(mover, SimEventType.BUMPED, slot=slot,
              src_hex=old_mover_loc, dst_hex=old_blocker_loc,
              dst_piece_id=blocker.id)
    
    return cost
```

Then the change in `_try_forward` is just a few lines — replace the blocked-by-occupied branch:

```python
# In _try_forward, replace this:
#   if nbr in occupied and nbr != piece.location:
#       self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
#       return 1.0

# With this:
if nbr in occupied and nbr != piece.location:
    blocker = next((p for p in self.alive if p.location == nbr), None)
    if blocker:
        bump_cost = self._try_bump(piece, blocker, occupied, slot)
        if bump_cost is not None:
            return bump_cost
    self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
    return 1.0
```

The nice thing about this design:

- **Rotations are free** (cost 0), so the bumped pawn's recovery doesn't burn its budget — it just needs one FORWARD action worth of budget next turn
- **If the mover stays put** (e.g. it was the last move of a march), the pawn's FORWARD will still be blocked, but it'll be facing the right way and ready to go once the blocker moves
- **If the mover has already moved on** by next turn, the pawn walks right back and rotates to resume its original plan

One thing to consider: should the bump cost extra for the mover (like +1 penalty), or just normal terrain cost? Normal cost feels right to me — the rank privilege *is* the benefit, no need to tax it further.

did I put these in correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Almost! Two issues with `_try_forward`:

**1. Missing `@patch` decorator and type annotation.** It's a standalone `def` right now, so it never gets attached to `FoodSimulator`. It needs:

```python
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None):
```

**2. Stray leading space.** There's a space before `def` that'll cause an `IndentationError`.

The bump logic inside it and `_try_bump` both look correct — the integration points are exactly right. Just fix the decorator and the indent on that one cell and you're good.

In [ ]:
@patch
def squad_chart(self: FoodSimulator, squad: Squad = None, figsize=(10, 4)):
    """Per-piece line chart of health & food over turns."""
    if not self.snapshots:
        print("No snapshots yet"); return

    pieces = squad.pieces if squad else self.pieces
    pid_set = {p.id for p in pieces}
    pid_names = {p.id: f"{p.name} ({p.piece_type.value})" for p in pieces}

    turns = [s['turn'] for s in self.snapshots]

    # Extract per-piece time series from existing snapshots
    health = {pid: [] for pid in pid_set}
    food   = {pid: [] for pid in pid_set}

    for snap in self.snapshots:
        states = snap.get('piece_states', {})
        for pid in pid_set:
            st = states.get(pid)
            health[pid].append(st['health'] if st else 0)
            food[pid].append(st['food']     if st else 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    for pid in pid_set:
        lbl = pid_names.get(pid, pid[:8])
        ax1.plot(turns, health[pid], 'o-', ms=3, lw=1.5, label=lbl)
        ax2.plot(turns, food[pid],   'o-', ms=3, lw=1.5, label=lbl)

    ax1.set_title('Health per turn', fontweight='bold', fontsize=10)
    ax1.set_xlabel('Turn'); ax1.set_ylim(bottom=0)
    ax1.legend(fontsize=7, loc='lower left')
    ax1.grid(axis='y', alpha=0.3)

    ax2.set_title('Food per turn', fontweight='bold', fontsize=10)
    ax2.set_xlabel('Turn'); ax2.set_ylim(bottom=0)
    ax2.legend(fontsize=7, loc='upper right')
    ax2.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    plt.show()


In [ ]:
#| export
def SquadFoodTrailOverlay(sim: 'FoodSimulator', squad: Squad = None,
                          min_opacity: float = 0.06,
                          max_opacity: float = 0.65,
                          sim_grid: HexGrid = None,
                          **kw) -> OverlaySpec:
    """Temporal food trail — pip circles sized by food %, arrows for shares, X for deaths.

    Opacity fades forward: earliest turns are faintest, most recent are boldest.
    Uses DiagramGlyphs for consistent styling with the rest of the overlay system.
    """

    def render(ctx) -> str:
        grid  = ctx.grid
        c2f   = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)

        piece_ids = ({p.id for p in squad.pieces} if squad
                     else {p.id for p in sim.pieces})

        snaps = sim.snapshots
        if not snaps:
            return ''
        max_turn = max(s['turn'] for s in snaps) or 1

        # Event log
        df = sim.df
        gives  = df[(df.event == 'gave')  & (df.piece_id.isin(piece_ids))]
        deaths = df[(df.event == 'died')  & (df.piece_id.isin(piece_ids))]

        r = grid.radius if hasattr(grid, 'radius') else 20

        # Build one DiagramGlyphs per flag we encounter, register styles once
        flag_glyphs: dict = {}
        def _get_glyphs(flag):
            key = id(flag) if flag else 'default'
            if key not in flag_glyphs:
                g = DiagramGlyphs(flag or CountryFlag.seaborn("husl", 1)[0], size=r * 0.7)
                g.register_styles(ctx.builder)
                # Register pip styles for food coloring
                for pct in range(0, 101, 5):
                    f = pct / 100
                    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
                    ctx.builder.add_style(StyleCSS(
                        f"pip_{pct}", fill=color, stroke=color,
                        stroke_width=0.8, opacity=0.85))
                flag_glyphs[key] = g
            return flag_glyphs[key]

        # Map piece_id → flag for glyph lookup
        piece_flags = {}
        src_pieces = squad.pieces if squad else sim.pieces
        for p in src_pieces:
            piece_flags[p.id] = p.flag

        def _render_idx(loc):
            ri = _map_point(loc, c2f) if c2f else loc
            return ri if 0 <= ri < len(grid.hexes) else -1

        parts = []

        # ── Per-turn food pips + share arrows ──
        for snap in snaps:
            t = snap['turn']
            alpha = min_opacity + (max_opacity - min_opacity) * (t / max_turn)
            states = snap.get('piece_states', {})

            # Food pips
            for pid, st in states.items():
                if pid not in piece_ids or st['health'] <= 0:
                    continue
                ri = _render_idx(st['location'])
                if ri < 0:
                    continue

                c = grid.hexes[ri].center
                cap = max(st['food_capacity'], 0.01)
                frac = np.clip(st['food'] / cap, 0, 1)

                glyphs = _get_glyphs(piece_flags.get(pid))
                pip_svg = glyphs.food_pip(MapCord(c.x, c.y), float(frac))
                parts.append(f'<g opacity="{alpha:.2f}">{pip_svg}</g>')

            # Give arrows for this turn
            turn_gives = gives[gives.turn == t]
            for _, row in turn_gives.iterrows():
                g_st = states.get(row.piece_id)
                r_st = states.get(row.dst_piece_id)
                if not g_st or not r_st:
                    continue
                gi = _render_idx(g_st['location'])
                ri2 = _render_idx(r_st['location'])
                if gi < 0 or ri2 < 0 or gi == ri2:
                    continue

                gc = grid.hexes[gi].center
                rc = grid.hexes[ri2].center
                glyphs = _get_glyphs(piece_flags.get(row.piece_id))
                arrow_svg = glyphs.transfer_arrow(
                    MapCord(gc.x, gc.y), MapCord(rc.x, rc.y))
                parts.append(f'<g opacity="{alpha:.2f}">{arrow_svg}</g>')

        # ── Death X markers (always prominent) ──
        for _, row in deaths.iterrows():
            ri = _render_idx(int(row.src_hex))
            if ri < 0:
                continue
            c = grid.hexes[ri].center
            glyphs = _get_glyphs(piece_flags.get(row.piece_id))
            parts.append(glyphs.death_marker(MapCord(c.x, c.y)))

        return '\n'.join(parts)

    return OverlaySpec("food_trail", render, priority=68)


In [ ]:
# Reset & apply food profiles (now that FOOD_RESERVE_TURNS exists)
  

# Find the queen
queen = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
queen.facing = 1  # West

# Queen marches west
myStuff.queen.instructions = InstructionList(
    [Instruction.FORWARD.value] * 10, cursor=0, patrol=False)

# Pawns: harvest + give patrol (stationary food stations)
for p in myStuff.queenGuard.pieces:
    if p.piece_type == PieceType.PAWN:
        facing_to_queen = _facing_toward(myStuff.grid, p.location, myStuff.queen.location)
        p.facing = facing_to_queen
        p.instructions = InstructionList(
            [Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.HARVEST.value, Instruction.GIVE.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value],
            cursor=0, patrol=True)

        #p.instructions = []
        for i in range(0, 30):
            p.instructions.rules.append(Instruction.FORWARD.value)

In [ ]:
sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=myStuff.food_tiers,
    pieces=list(myStuff.queenGuard.pieces),
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=18)
sim.squad_chart(myStuff.queenGuard)


In [ ]:

myStuff.clearTerr()
region = HexRegion(hexes=set(myStuff.terr.ring(myStuff.queen.location, 7)), hexGrid=myStuff.grid)
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(scale=5),
    KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard, sim_grid=myStuff.grid),
    #SquadPlanOverlay([queenGuard], num_turns=8),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    region=region,
    debug = not showDemo
)

In [ ]:
show(myStuff.queen)

GroupedPieceList(myStuff.queenGuard.pieces,grid=myStuff.grid)

In [ ]:
show(myStuff.queenGuard)

In [ ]:
sim.squad_chart(myStuff.queenGuard)


In [ ]:
# === Supply Line Concept Demo ===
myStuff = GameParts()  # reset


soil = SoilSystem.from_plates(myStuff.terr, [])

fy = FoodYield(myStuff.terr, myStuff.basin)
fy.compute()
#show(fy)
myStuff.fy = fy

queen = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
pawns = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.PAWN]
queen.facing = 1  # march west

food_tiers = myStuff.fy.tiers
countries = myStuff.terr.fields.get('country')
elevs = myStuff.terr.elevations
grid = myStuff.grid

# --- Trace queen's march path (straight line, stop at impassable) ---
path = [queen.location]
loc = queen.location
for _ in range(30):
    d = HexPosition.directions()[queen.facing % 6]
    nbr = grid.hexposition_to_index(d, loc)
    if (nbr < 0 or nbr >= len(elevs) or nbr in grid.invalidRegion
        or elevs[nbr] < 1): break
    if countries is not None and countries[nbr] < 0: break
    path.append(nbr)
    loc = nbr

print(f"March path: {len(path)} hexes")

# Queen: forward + harvest every 3rd step so she grazes on the move
qrules = []
for i in range(len(path) - 1):
    qrules.append(Instruction.FORWARD.value)
    if (i + 1) % 3 == 0:
        qrules.append(Instruction.HARVEST.value)
queen.instructions = InstructionList(qrules, cursor=0, patrol=False)

# --- Teleport pawns to supply stations alongside the path ---
path_set = set(path)
spacing = max(1, (len(path) - 1) // len(pawns))

for i, pawn in enumerate(pawns):
    target_i = min(2 + i * spacing, len(path) - 1)
    target_hex = path[target_i]

    # Find best adjacent food tile NOT on the march path
    best, best_tier = None, 0
    for hp in HexPosition.origin().ring(1):
        adj = grid.hexposition_to_index(hp, target_hex)
        if (adj < 0 or adj >= len(elevs) or adj in grid.invalidRegion
            or elevs[adj] < 1 or adj in path_set): continue
        tier = int(food_tiers[adj])
        if tier > best_tier: best, best_tier = adj, tier

    if best is None: best = target_hex  # fallback

    pawn.location = best          # teleport (concept demo)
    pawn.facing = _facing_toward(grid, best, target_hex)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    print(f"  {pawn.name} → hex {best} (tier {best_tier}), feeding path[{target_i}]={target_hex}")

# --- Run ---
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=list(myStuff.queenGuard.pieces),
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=30)
sim.summary()


In [ ]:
sim.squad_chart(myStuff.queenGuard)


## Recording

I think we want the food simulator to generate a pandas dataframe of the actions.

# Stability (Our first atttempt at Pyomo)

In [ ]:
#| export
def _solver_ok(result) -> bool:
    """Robust optimality check across classic and appsi Pyomo interfaces."""
    try:
        tc = result.termination_condition          # appsi interface
    except AttributeError:
        tc = result.solver.termination_condition   # classic interface
    return 'optimal' in str(tc).lower()


@patch
def feed_cover(self: Squad, food_tiers: np.ndarray,
               objective: str = 'attack',
               max_feeds: int = 99) -> dict:
    """Optimal feeder assignment using Pyomo."""
    alive = self.alive
    if not alive:
        return dict(feeders=[], free=[], total_production=0,
                    total_diet=0, surplus=0, feasible=False)

    m = pyo.ConcreteModel()
    N = len(alive)
    m.I = pyo.RangeSet(0, N - 1)
    m.x = pyo.Var(m.I, domain=pyo.Binary)

    production = []
    for p in alive:
        loc = p.location
        tier = int(food_tiers[loc]) if loc is not None and 0 <= loc < len(food_tiers) else 0
        feed_tries = max(min(p.move_strength - 1, max_feeds), 0)
        production.append(feed_tries * p.harvest_strength * tier)

    total_diet = sum(p.diet for p in alive)
    max_prod = sum(production)

    if max_prod < total_diet:
        return dict(feeders=list(alive), free=[], total_production=max_prod,
                    total_diet=total_diet, surplus=max_prod - total_diet,
                    feasible=False,
                    production_per_feeder={p.name: production[i] for i, p in enumerate(alive)})

    m.food = pyo.Constraint(
        expr=sum(m.x[i] * production[i] for i in m.I) >= total_diet)

    values = ([getattr(p, 'attack_strength', 0) for p in alive]
              if objective == 'attack'
              else [p.food_capacity for p in alive])

    m.obj = pyo.Objective(
        expr=sum(values[i] * m.x[i] for i in m.I),
        sense=pyo.minimize)

    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)          # let it load automatically
    ok = _solver_ok(result)

    if not ok:
        return dict(feeders=list(alive), free=[], total_production=max_prod,
                    total_diet=total_diet, surplus=max_prod - total_diet,
                    feasible=False,
                    production_per_feeder={p.name: production[i] for i, p in enumerate(alive)})

    feeders = [alive[i] for i in m.I if pyo.value(m.x[i]) > 0.5]
    free    = [alive[i] for i in m.I if pyo.value(m.x[i]) <= 0.5]
    total_prod = sum(production[i] * pyo.value(m.x[i]) for i in m.I)

    return dict(
        feeders=feeders, free=free,
        total_production=total_prod, total_diet=total_diet,
        surplus=total_prod - total_diet, feasible=ok,
        production_per_feeder={p.name: production[alive.index(p)] for p in feeders},
    )


In [ ]:
#| export
@patch
def march_range(self: Squad, leader_path: TroopPath,
                grid: HexGrid, elevations: np.ndarray,
                formation: FormationType = None,  # None = broken formation
                countries: np.ndarray = None,
                food_tiers: np.ndarray = None,
                min_food_tier: int = 2,
                leader: Piece = None) -> dict:
    """Farthest waypoint along leader_path where ALL pieces arrive with food >= 0.
    
    formation=None:          broken — everyone pathfinds to waypoint hex
    formation=FormationType: formed — followers pathfind to formation slots
    """
    alive = self.alive
    if not alive or not leader_path:
        return dict(waypoint=0, hex_idx=leader_path[0] if leader_path else -1,
                    feasible=False, arrivals=pd.DataFrame(), paths={})

    if leader is None:
        leader = self.by_rank()[0]
    followers = [p for p in alive if p.id != leader.id]

    # Cumulative cost along leader_path
    cum_cost = [0.0]
    for k in range(1, len(leader_path)):
        cum_cost.append(cum_cost[-1] + _move_cost(elevations, leader_path[k-1], leader_path[k]))

    # Scan backwards from farthest — first feasible wins
    for j in range(len(leader_path) - 1, 0, -1):
        wp = leader_path[j]

        # Feedability check: waypoint or ring-1 must have food
        if food_tiers is not None:
            best_tier = int(food_tiers[wp]) if wp < len(food_tiers) else 0
            for hp in HexPosition.origin().ring(1):
                nb = grid.hexposition_to_index(hp, wp)
                if 0 <= nb < len(food_tiers):
                    best_tier = max(best_tier, int(food_tiers[nb]))
            if best_tier < min_food_tier:
                continue

        # Leader arrival
        leader_turns = math.ceil(cum_cost[j] / max(leader.move_strength, 1))
        leader_food = leader.food - leader.diet * leader_turns
        if leader_food < 0:
            continue

        # Arrival facing for formation slots
        arrive_facing = _facing_toward(grid, leader_path[j-1], leader_path[j])

        # Follower targets
        if formation is not None and followers:
            offsets = _formation_offsets(formation, len(followers), arrive_facing)
            targets = []
            for off in offsets:
                idx = grid.hexposition_to_index(off, wp)
                ok = (0 <= idx < len(elevations)
                      and idx not in grid.invalidRegion
                      and elevations[idx] > 0
                      and (countries is None or countries[idx] >= 0))
                targets.append(idx if ok else None)
        else:
            targets = [wp] * len(followers)

        # Check each follower can reach their target alive
        all_ok = True
        follower_paths = {}
        for i, piece in enumerate(followers):
            target = targets[i] if i < len(targets) else None
            if target is None:
                all_ok = False; break
            tp = piece.pathfind(target, grid, elevations, countries)
            if not tp:
                all_ok = False; break
            turns = math.ceil(tp.cost / max(piece.move_strength, 1))
            if piece.food - piece.diet * turns < 0:
                all_ok = False; break
            follower_paths[piece.id] = tp

        if not all_ok:
            continue

        # Build result
        leader_sub = TroopPath(leader_path.hexes[:j+1], cum_cost[j])
        follower_paths[leader.id] = leader_sub

        rows = [dict(name=leader.name, type=leader.piece_type.name,
                     turns=leader_turns, food_left=round(leader_food, 2),
                     target_hex=wp)]
        for i, piece in enumerate(followers):
            tp = follower_paths[piece.id]
            turns = math.ceil(tp.cost / max(piece.move_strength, 1))
            rows.append(dict(name=piece.name, type=piece.piece_type.name,
                             turns=turns,
                             food_left=round(piece.food - piece.diet * turns, 2),
                             target_hex=targets[i]))

        return dict(waypoint=j, hex_idx=wp, path_cost=round(cum_cost[j], 2),
                    leader_path=leader_sub, feasible=True,
                    arrivals=pd.DataFrame(rows), paths=follower_paths,
                    formation=formation)

    return dict(waypoint=0, hex_idx=leader_path[0],
                feasible=False, arrivals=pd.DataFrame(), paths={})

In [ ]:
#| export
@patch
def leapfrog_plan(self: Squad, leader_path: TroopPath,
                  grid: HexGrid, elevations: np.ndarray,
                  formation: FormationType = None,
                  countries: np.ndarray = None,
                  food_tiers: np.ndarray = None,
                  min_food_tier: int = 2,
                  recovery_turns: int = 4,
                  max_legs: int = 20,
                  leader: Piece = None) -> list[dict]:
    """Chain march_range + feed_cover into travel/feed/travel/feed legs.
    
    Before each leg:  pre_march_distribute — bishop tops up everyone.
    After each leg:   squad_recover — harvest + surplus sharing at waypoint.
    Then redistribute again before the next departure.
    """
    if leader is None:
        leader = self.by_rank()[0]

    legs = []
    pos = 0

    for _ in range(max_legs):
        if pos >= len(leader_path) - 1:
            break

        # ── Phase 0: Bishop tops up squad before departure ──
        pre_march_distribute(self)

        # ── Phase 1: Find farthest reachable waypoint ──
        sub_hexes = leader_path.hexes[pos:]
        sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)

        leg = self.march_range(sub_path, grid, elevations,
                               formation=formation, countries=countries,
                               food_tiers=food_tiers,
                               min_food_tier=min_food_tier, leader=leader)

        if not leg['feasible'] or leg['waypoint'] == 0:
            break

        # ── Phase 2: Deduct travel cost from each piece ──
        wp_hex = leg['hex_idx']
        for p in self.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)

        # ── Phase 3: Find best food tier near the waypoint ──
        best_tier = int(food_tiers[wp_hex]) if wp_hex < len(food_tiers) else 0
        for hp in HexPosition.origin().ring(1):
            nb = grid.hexposition_to_index(hp, wp_hex)
            if 0 <= nb < len(food_tiers):
                best_tier = max(best_tier, int(food_tiers[nb]))

        # ── Phase 4: Recover at waypoint (harvest + share) ──
        for p in self.alive:
            p.location = wp_hex
        squad_recover(self, best_tier, recovery_turns)

        # ── Phase 5: Redistribute after recovery (prep for next leg) ──
        pre_march_distribute(self)

        # ── Phase 6: Record feed cover snapshot ──
        cover = self.feed_cover(food_tiers, objective='attack')
        leg['feed_cover'] = cover
        leg['recovery_tier'] = best_tier
        leg['leg_number'] = len(legs)
        legs.append(leg)

        pos += leg['waypoint']

    return legs


In [ ]:
#| export
@patch
def march_analysis(self: Squad, leader_path: TroopPath,
                   grid: HexGrid, elevations: np.ndarray,
                   formation: FormationType = FormationType.COLUMN,
                   countries: np.ndarray = None,
                   food_tiers: np.ndarray = None,
                   min_food_tier: int = 2,
                   leader: Piece = None) -> dict:
    """Compare formation vs broken formation march range.
    
    Returns both results + the 'bonus' hexes gained by breaking formation.
    """
    formed = self.march_range(leader_path, grid, elevations,
                              formation=formation, countries=countries,
                              food_tiers=food_tiers,
                              min_food_tier=min_food_tier, leader=leader)
    broken = self.march_range(leader_path, grid, elevations,
                              formation=None, countries=countries,
                              food_tiers=food_tiers,
                              min_food_tier=min_food_tier, leader=leader)
    return {
        'formation': formed,
        'broken': broken,
        'formation_type': formation,
        'formation_range': formed['waypoint'],
        'broken_range': broken['waypoint'],
        'range_bonus': broken['waypoint'] - formed['waypoint'],
    }


In [ ]:
myStuff = GameParts()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers
grid, elevs = myStuff.grid, myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')
queen = myStuff.queenGuard.by_rank()[0]

# Queen pathfinds west as far as she can
far_west = queen.location
for _ in range(30):
    d = HexPosition.directions()[1]  # west
    nb = grid.hexposition_to_index(d, far_west)
    if nb < 0 or nb >= len(elevs) or elevs[nb] < 1: break
    if countries is not None and countries[nb] < 0: break
    far_west = nb

route = queen.pathfind(far_west, grid, elevs, countries)
print(f"Leader route: {len(route)} hexes, cost {route.cost:.0f}")

# Compare formation vs broken
analysis = myStuff.queenGuard.march_analysis(
    route, grid, elevs,
    formation=FormationType.COLUMN,
    countries=countries,
    food_tiers=food_tiers,
    min_food_tier=2)

print(f"\nFormation range: {analysis['formation_range']} hexes (COLUMN)")
print(f"Broken range:    {analysis['broken_range']} hexes")
print(f"Bonus for breaking formation: +{analysis['range_bonus']} hexes")

if analysis['formation']['feasible']:
    print(f"\nFormation arrivals:")
    print(analysis['formation']['arrivals'].to_string(index=False))

if analysis['broken']['feasible']:
    print(f"\nBroken arrivals:")
    print(analysis['broken']['arrivals'].to_string(index=False))

# Leapfrog in broken formation
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=None, countries=countries,
    food_tiers=food_tiers, recovery_turns=4)

for leg in legs:
    fc = leg['feed_cover']
    print(f"\nLeg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(cost {leg['path_cost']})")
    print(f"  Feeders: {[p.name for p in fc['feeders']]}")
    print(f"  Free:    {[p.name for p in fc['free']]}")

total = sum(l['waypoint'] for l in legs)
print(f"\nTotal coverage: {total}/{len(route)} hexes in {len(legs)} legs")


In [ ]:
@patch
def farthest_route(self: Terrain, start: int, 
                   countries: np.ndarray = None) -> TroopPath:
    """Dijkstra flood from start → TroopPath to the farthest reachable hex."""
    if not hasattr(self, '_route_cache'): self._route_cache = {}
    if start in self._route_cache: return self._route_cache[start]
    
    grid, elevs = self.hexGrid, self.elevations
    
    pq = [(0.0, start)]
    best = {start: 0.0}
    parent = {start: None}
    visited = set()
    
    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited.add(cur)
        for nb in grid.neighborsOf(cur):
            if nb in visited or nb in grid.invalidRegion: continue
            if elevs[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            nc = cost + _move_cost(elevs, cur, nb)
            if nc < best.get(nb, float('inf')):
                best[nb] = nc
                parent[nb] = cur
                heapq.heappush(pq, (nc, nb))
    
    far = max(best, key=best.get)
    
    # Reconstruct path from parent chain
    path = []
    node = far
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    
    route = TroopPath(path, best[far])
    self._route_cache[start] = route
    return route


In [ ]:
def demoSquad(numQueens=1, numBishops=1, numPawns=5):
    myStuff = GameParts()
    
    fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
    food_tiers = fy.tiers
    grid, elevs = myStuff.grid, myStuff.terr.elevations
    countries = myStuff.terr.fields.get('country')

    # ── Rebuild the squad ──
    myStuff.queenGuard.pieces.clear()
    for _ in range(numQueens):
        q = myStuff.home.recruit(PieceType.QUEEN, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(q)
    for _ in range(numPawns):
        p = myStuff.home.recruit(PieceType.PAWN, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(p)
    for _ in range(numBishops):
        p = myStuff.home.recruit(PieceType.BISHOP, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(p)

    myStuff.home.place_squad_spiral(myStuff.queenGuard, myStuff.terr)

    # ── Provision the squad before departure ──
    for p in myStuff.queenGuard.alive:
        p.food = p.food_capacity

    total_diet = sum(p.diet for p in myStuff.queenGuard.alive)
    total_food = sum(p.food for p in myStuff.queenGuard.alive)
    print(f"Squad: {numQueens}Q + {numBishops}B + {numPawns}P")
    print(f"Total diet/turn: {total_diet:.1f}  Total food: {total_food:.1f}")
    for p in myStuff.queenGuard.alive:
        print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
              f"diet={p.diet:.1f} food={p.food:.1f}/{p.food_capacity:.1f} "
              f"speed={p.move_strength}")

    # ── Route to the coast ──
    leader = myStuff.queenGuard.by_rank()[0]
    route = myStuff.terr.farthest_route(leader.location, countries)
    print(f"\nRoute: {len(route)} hexes, cost {route.cost:.0f}")

    # ── Pre-check: starting feed cover ──
    start_cover = myStuff.queenGuard.feed_cover(food_tiers, objective='attack')
    print(f"\nStarting feed cover: feasible={start_cover['feasible']}")
    print(f"  Production: {start_cover['total_production']:.1f} vs Diet: {start_cover['total_diet']:.1f}")
    print(f"  Feeders: {[p.name for p in start_cover['feeders']]}")
    print(f"  Free:    {[p.name for p in start_cover['free']]}")

    # ── Leapfrog to the coast ──
    legs = myStuff.queenGuard.leapfrog_plan(
        route, grid, elevs,
        formation=None, countries=countries,
        food_tiers=food_tiers,
        min_food_tier=1,
        recovery_turns=6,
        max_legs=50,
        leader=leader)

    total = sum(l['waypoint'] for l in legs)
    pct = total / max(len(route) - 1, 1) * 100

    for leg in legs:
        fc = leg.get('feed_cover', {})
        print(f"Leg {leg['leg_number']:2d}: +{leg['waypoint']:2d} hexes → hex {leg['hex_idx']:4d} "
              f"(cost {leg['path_cost']:5.1f}) | tier={leg.get('recovery_tier', '?')} "
              f"feeders={len(fc.get('feeders', []))} free={len(fc.get('free', []))}")

    if total >= len(route) - 1:
        print(f"\n🏖 REACHED THE COAST in {len(legs)} legs!")
    else:
        print(f"\n⛔ STUCK after {len(legs)} legs — {total}/{len(route)-1} hexes ({pct:.0f}%)")
        for p in myStuff.queenGuard.alive:
            print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
                  f"food={p.food:.1f}/{p.food_capacity:.1f} diet={p.diet:.1f}")

    print(f"\nTotal: {total}/{len(route)-1} hexes in {len(legs)} legs ({pct:.0f}%)")
    for p in myStuff.queenGuard.alive:
        print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
              f"hp={p.health:.0f} food={p.food:.1f}/{p.food_capacity:.1f} diet={p.diet:.1f}")

    # ── Sim for visualization ──
    myStuff.food_tiers = food_tiers
    myStuff.fy = fy
    sim = FoodSimulator(
        grid=grid, elevations=elevs, food_tiers=food_tiers,
        pieces=list(myStuff.queenGuard.pieces),
        countries=countries,
        squads=[myStuff.queenGuard],
        callbacks={myStuff.queenGuard.id: food_triage},
    )
    sim.run(num_turns=max(30, len(route)))

    return SimpleNamespace(myStuff=myStuff, route=route, legs=legs,
                           leader=leader, sim=sim)


In [ ]:
res = demoSquad()
ms = res.myStuff

center = res.leader.location
radius = max(10, len(res.route) // 2)
region = HexRegion(hexes=set(ms.terr.ring(center, radius)), hexGrid=ms.grid)

ms.clearTerr()
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(scale=5),
    KingdomNamesOverlay(),
    PieceOverlay([ms.queenGuard]),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    SquadFoodTrailOverlay(res.sim, ms.queenGuard, sim_grid=ms.grid),
    terrain=ms.terr,
    basins=ms.basin,
    region=region,
    debug=not showDemo
)


In [ ]:
from types import SimpleNamespace

res = demoSquad()
ms = res.myStuff

# Center on wherever the leader ended up, wide enough to see the whole route
center = res.leader.location
radius = max(10, len(res.route) // 2)
region = HexRegion(hexes=set(ms.terr.ring(center, radius)), hexGrid=ms.grid)

ms.clearTerr()
TerrainDisplay(
    CreamOverlay(),
    SettlementOverlay(),
    KingdomNamesOverlay(),
    PieceOverlay([ms.queenGuard]),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    # SquadPlanOverlay([ms.queenGuard], num_turns=8),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard, sim_grid=myStuff.grid),
    terrain=ms.terr,
    basins=ms.basin,
    region=region,
    debug=not showDemo
)


Do we need to redesign path planning and put in something like squad_recover into our algorithm

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — the planning pipeline has a gap right now. Here's the diagnosis:

**The problem:** `march_range` checks "can each piece reach the waypoint with food ≥ 0" — but it never calls `pre_march_distribute` first. So the bishop is sitting on 178.7 food while pawns starve after 3 hexes. The bishop's whole *raison d'être* is bankrolling the squad, but the planner doesn't let it.

**What's missing in the chain:**

```
Current:  march_range(raw food) → fail → leapfrog gives up

Needed:   pre_march_distribute(bishop tops up everyone)
          → march_range(boosted food) → succeed
          → squad_recover(harvest + share at waypoint)
          → pre_march_distribute again
          → next leg
```

Two things to wire in:

1. **Before each leg** — call `pre_march_distribute` so the bishop fills up pawns and the queen. With 178.7 food the bishop can top up all 5 pawns (need 5 × 8.7 = 43.5) and still have 130+ left for the queen.

2. **At each waypoint** — call `squad_recover(squad, food_tier, recovery_turns)` instead of the simplified `p.food += p.diet * recovery_turns` that `leapfrog_plan` currently does. This gives you the proper overflow-to-bishop / deficit-from-bishop mechanics.

The fix in `leapfrog_plan` would look roughly like:

```python
# Before each leg:
pre_march_distribute(self)

# At each waypoint (replace the current simple recovery):
for p in self.alive:
    p.location = leg['hex_idx']
best_tier = int(food_tiers[leg['hex_idx']])
# check ring-1 for better food
for hp in HexPosition.origin().ring(1):
    nb = grid.hexposition_to_index(hp, leg['hex_idx'])
    if 0 <= nb < len(food_tiers):
        best_tier = max(best_tier, int(food_tiers[nb]))
squad_recover(self, best_tier, recovery_turns)
pre_march_distribute(self)  # redistribute before next leg
```

This should unstick the 1Q+1B+5P case immediately — the bishop can fund roughly **94 turns** of its own travel plus dozens of pawn top-ups. The 1Q+5P (no bishop) case will still struggle, which is *by design* — that's what makes the bishop matter.

You don't need Pyomo for this fix. The temporal optimization model is for the *next* level — optimizing *where* to place feeders and *when* to harvest. But the basic "bishop shares before departure, squad recovers at waypoints" loop is pure imperative code and should be integrated into `leapfrog_plan` right now.

can you rewrite leapfrog_plan so that it does these things?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def leapfrog_plan(self: Squad, leader_path: TroopPath,
                  grid: HexGrid, elevations: np.ndarray,
                  formation: FormationType = None,
                  countries: np.ndarray = None,
                  food_tiers: np.ndarray = None,
                  min_food_tier: int = 2,
                  recovery_turns: int = 4,
                  max_legs: int = 20,
                  leader: Piece = None) -> list[dict]:
    """Chain march_range + feed_cover into travel/feed/travel/feed legs.
    
    Before each leg:  pre_march_distribute — bishop tops up everyone.
    After each leg:   squad_recover — harvest + surplus sharing at waypoint.
    Then redistribute again before the next departure.
    """
    if leader is None:
        leader = self.by_rank()[0]

    legs = []
    pos = 0

    for _ in range(max_legs):
        if pos >= len(leader_path) - 1:
            break

        # ── Phase 0: Bishop tops up squad before departure ──
        pre_march_distribute(self)

        # ── Phase 1: Find farthest reachable waypoint ──
        sub_hexes = leader_path.hexes[pos:]
        sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)

        leg = self.march_range(sub_path, grid, elevations,
                               formation=formation, countries=countries,
                               food_tiers=food_tiers,
                               min_food_tier=min_food_tier, leader=leader)

        if not leg['feasible'] or leg['waypoint'] == 0:
            break

        # ── Phase 2: Deduct travel cost from each piece ──
        wp_hex = leg['hex_idx']
        for p in self.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)

        # ── Phase 3: Find best food tier near the waypoint ──
        best_tier = int(food_tiers[wp_hex]) if wp_hex < len(food_tiers) else 0
        for hp in HexPosition.origin().ring(1):
            nb = grid.hexposition_to_index(hp, wp_hex)
            if 0 <= nb < len(food_tiers):
                best_tier = max(best_tier, int(food_tiers[nb]))

        # ── Phase 4: Recover at waypoint (harvest + share) ──
        for p in self.alive:
            p.location = wp_hex
        squad_recover(self, best_tier, recovery_turns)

        # ── Phase 5: Redistribute after recovery (prep for next leg) ──
        pre_march_distribute(self)

        # ── Phase 6: Record feed cover snapshot ──
        cover = self.feed_cover(food_tiers, objective='attack')
        leg['feed_cover'] = cover
        leg['recovery_tier'] = best_tier
        leg['leg_number'] = len(legs)
        legs.append(leg)

        pos += leg['waypoint']

    return legs
```

The key changes:

1. **`pre_march_distribute(self)` at top of each iteration** — bishop fills pawns and queen from its 178.7 reserve before anyone moves. Pawns go from 8.7 → 8.7 (already full), queen stays topped up.

2. **`squad_recover(self, best_tier, recovery_turns)` at waypoint** — replaces the naive `p.food += p.diet * recovery_turns`. Now pawns harvest, overflow goes to bishop (the bank), deficit pieces draw from bishop first. This is the actual food economy loop.

3. **`pre_march_distribute(self)` again after recovery** — so the squad departs the waypoint with everyone topped up from the bishop's reserves.

4. **Best-tier search in ring-1** — pieces don't have to stand *on* the waypoint to recover; they use the best food nearby.

The travel cost deduction (Phase 2) is kept separate from recovery (Phase 4) so the food math is clear: spend, then earn.

Shouldn't with leapfrog we should be getting somewhere?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, absolutely — with a bishop carrying 178.7 food and `squad_recover` at each waypoint, you should be covering serious ground. The fact that you're getting 0 legs means `march_range` is rejecting **every** waypoint before it even tries. The culprit is the `min_food_tier` filter:

```python
if best_tier < min_food_tier:
    continue  # ← skipping waypoints on barren terrain
```

`farthest_route` finds the longest path, which often cuts through mountains — food tier 0 everywhere. Even with `min_food_tier=1`, if the early route hexes are all barren (no tier ≥ 1 within ring-1), then *every* waypoint fails and `march_range` returns `waypoint=0`.

But this is conceptually wrong. The squad doesn't need food **at every waypoint** — it needs food **at recovery stops**. Between stops, the bishop bankrolls the march. The fix: separate "can I reach this hex" from "can I recover here."

Two approaches:

**Quick fix** — just set `min_food_tier=0` in the `leapfrog_plan` call so march_range never filters. Recovery at tier-0 will be bad (zero harvest), but the bishop's reserves will still carry the squad.

**Better fix** — remove the food tier check from `march_range` entirely, and instead have `leapfrog_plan` find the best recovery hex *near* the waypoint after arrival:

```python
# In leapfrog_plan, after march_range succeeds:
# Instead of recovering at wp_hex, find the best food nearby
best_tier, best_hex = 0, wp_hex
for hp in HexPosition.origin().ring(1) + [HexPosition.origin()]:
    nb = grid.hexposition_to_index(hp, wp_hex)
    if 0 <= nb < len(food_tiers) and int(food_tiers[nb]) > best_tier:
        best_tier, best_hex = int(food_tiers[nb]), nb

# Recover at the best nearby hex (might be tier 0 — bishop covers it)
for p in self.alive:
    p.location = best_hex
squad_recover(self, best_tier, recovery_turns)
```

And in `march_range`, delete the food tier gate entirely — it's a march planner, not a recovery planner. The leapfrog wrapper handles recovery.

Should path planning come up with a better route of way stations. Maybe we want to run not at farthest point but farthest reachable. It is a tricky thing because there are two thing 1 how far you go if you drain the tank and how far you go and be substainable.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're hitting on the key insight. The current system does "plan route, then try to survive it" — but what you really want is "plan route *around* the food." Two very different problems.

**The two modes you identified:**

| Mode | Question | Bishop role |
|------|----------|-------------|
| **Sprint** | How far before everyone dies? | Burn the tank, no recovery |
| **Sustainable** | How far can we keep going indefinitely? | Refill at waystations, bishop smooths gaps |

The sprint range is a ceiling — useful for "can we reach that city before we starve." The sustainable range is the real strategic planner.

**The fix: waystation-aware routing.** Instead of Dijkstra-to-farthest then leapfrog, flip it:

1. **Build a waystation graph** — cluster all hexes with food tier ≥ N into "rest stops"
2. **Edges** connect waystations that are within the squad's one-leg fuel budget (bishop-boosted)
3. **Route through the graph** toward the destination — each hop is one leg of the leapfrog

This is the classic "gas station problem": given fuel capacity, find the path through refueling stations that gets you farthest (or reaches the goal in fewest stops).

The greedy version is simple and probably good enough:

```python
def waystation_route(squad, start, goal, grid, elevations, food_tiers,
                     countries=None, min_tier=3, recovery_turns=4):
    """Greedy hop: from current position, find the farthest reachable
    waystation that's closest to the goal. Recover there. Repeat."""
    
    legs = []
    pos = start
    
    while pos != goal:
        pre_march_distribute(squad)
        
        # 1. Find all waystations within fuel range
        #    (Dijkstra flood from pos, capped by squad's total food budget)
        budget = min(p.food / p.diet for p in squad.alive)  # turns of food
        max_cost = budget * min(p.move_strength for p in squad.alive)
        
        candidates = _flood_to_budget(pos, max_cost, grid, elevations, countries)
        
        # 2. Filter to good food tiles
        waystations = [h for h in candidates 
                       if _best_food_nearby(h, grid, food_tiers) >= min_tier]
        
        if not waystations:
            # No sustainable option — sprint toward goal as far as we can
            break
        
        # 3. Pick the one closest to the goal (progress-maximizing)
        best = min(waystations, 
                   key=lambda h: _hex_distance(h, goal, grid))
        
        # 4. March there, recover
        leg_path = _pathfind(pos, best, grid, elevations, countries)
        # ... deduct food, squad_recover, record leg ...
        
        pos = best
    
    return legs
```

This naturally handles both modes: if there are waystations, it's sustainable. If the only candidate is the goal itself (no food), it's a sprint. And it never plans a route through a food desert when there's a fertile valley one hex over.

Want to build out `_flood_to_budget` first, or should we start with the waystation identification (clustering good-food hexes)?

can you rewrite demoSquad so it uses leapfrog_plan?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

```python
def demoSquad(numQueens=1, numBishops=1, numPawns=5):
    myStuff = GameParts()
    
    fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
    food_tiers = fy.tiers
    grid, elevs = myStuff.grid, myStuff.terr.elevations
    countries = myStuff.terr.fields.get('country')

    # ── Rebuild the squad ──
    myStuff.queenGuard.pieces.clear()
    for _ in range(numQueens):
        q = myStuff.home.recruit(PieceType.QUEEN, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(q)
    for _ in range(numPawns):
        p = myStuff.home.recruit(PieceType.PAWN, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(p)
    for _ in range(numBishops):
        p = myStuff.home.recruit(PieceType.BISHOP, flag=myStuff.queenGuard.flag)
        myStuff.queenGuard.pieces.append(p)

    myStuff.home.place_squad_spiral(myStuff.queenGuard, myStuff.terr)

    # ── Provision the squad before departure ──
    for p in myStuff.queenGuard.alive:
        p.food = p.food_capacity

    total_diet = sum(p.diet for p in myStuff.queenGuard.alive)
    total_food = sum(p.food for p in myStuff.queenGuard.alive)
    print(f"Squad: {numQueens}Q + {numBishops}B + {numPawns}P")
    print(f"Total diet/turn: {total_diet:.1f}  Total food: {total_food:.1f}")
    for p in myStuff.queenGuard.alive:
        print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
              f"diet={p.diet:.1f} food={p.food:.1f}/{p.food_capacity:.1f} "
              f"speed={p.move_strength}")

    # ── Route to the coast ──
    leader = myStuff.queenGuard.by_rank()[0]
    route = myStuff.terr.farthest_route(leader.location, countries)
    print(f"\nRoute: {len(route)} hexes, cost {route.cost:.0f}")

    # ── Pre-check: starting feed cover ──
    start_cover = myStuff.queenGuard.feed_cover(food_tiers, objective='attack')
    print(f"\nStarting feed cover: feasible={start_cover['feasible']}")
    print(f"  Production: {start_cover['total_production']:.1f} vs Diet: {start_cover['total_diet']:.1f}")
    print(f"  Feeders: {[p.name for p in start_cover['feeders']]}")
    print(f"  Free:    {[p.name for p in start_cover['free']]}")

    # ── Leapfrog to the coast ──
    legs = myStuff.queenGuard.leapfrog_plan(
        route, grid, elevs,
        formation=None, countries=countries,
        food_tiers=food_tiers,
        min_food_tier=1,
        recovery_turns=6,
        max_legs=50,
        leader=leader)

    total = sum(l['waypoint'] for l in legs)
    pct = total / max(len(route) - 1, 1) * 100

    for leg in legs:
        fc = leg.get('feed_cover', {})
        print(f"Leg {leg['leg_number']:2d}: +{leg['waypoint']:2d} hexes → hex {leg['hex_idx']:4d} "
              f"(cost {leg['path_cost']:5.1f}) | tier={leg.get('recovery_tier', '?')} "
              f"feeders={len(fc.get('feeders', []))} free={len(fc.get('free', []))}")

    if total >= len(route) - 1:
        print(f"\n🏖 REACHED THE COAST in {len(legs)} legs!")
    else:
        print(f"\n⛔ STUCK after {len(legs)} legs — {total}/{len(route)-1} hexes ({pct:.0f}%)")
        for p in myStuff.queenGuard.alive:
            print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
                  f"food={p.food:.1f}/{p.food_capacity:.1f} diet={p.diet:.1f}")

    print(f"\nTotal: {total}/{len(route)-1} hexes in {len(legs)} legs ({pct:.0f}%)")
    for p in myStuff.queenGuard.alive:
        print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
              f"hp={p.health:.0f} food={p.food:.1f}/{p.food_capacity:.1f} diet={p.diet:.1f}")

    # ── Sim for visualization ──
    myStuff.food_tiers = food_tiers
    myStuff.fy = fy
    sim = FoodSimulator(
        grid=grid, elevations=elevs, food_tiers=food_tiers,
        pieces=list(myStuff.queenGuard.pieces),
        countries=countries,
        squads=[myStuff.queenGuard],
        callbacks={myStuff.queenGuard.id: food_triage},
    )
    sim.run(num_turns=max(30, len(route)))

    return SimpleNamespace(myStuff=myStuff, route=route, legs=legs,
                           leader=leader, sim=sim)
```

The big change: the entire hand-rolled `for leg_n in range(max_legs):` loop is gone, replaced by the single `leapfrog_plan` call. That means:

- **`pre_march_distribute`** runs before each leg (bishop tops up everyone)
- **`squad_recover`** runs at each waypoint (proper harvest + share)  
- **`pre_march_distribute`** runs again after recovery (prepped for next leg)
- **`min_food_tier=1`** so it doesn't reject marginal terrain

Shouldn't it have been reachable? Is the queen pinned in by the pawns?

In [ ]:
demoSquad(1,0)

The queen dies without help

In [ ]:
!pip install optuna optuna-dashboard

In [ ]:
import optuna

def balance_objective(trial):
    """One trial = one set of piece stats → score across all scenarios."""
    
    # ── Suggest parameters ──
    queen_diet     = trial.suggest_float('queen_diet', 2.0, 8.0)
    queen_capacity = trial.suggest_float('queen_capacity', 8.0, 20.0)
    queen_harvest  = trial.suggest_float('queen_harvest', 0.5, 2.0)
    
    bishop_diet     = trial.suggest_float('bishop_diet', 1.0, 4.0)
    bishop_capacity = trial.suggest_float('bishop_capacity', 6.0, 14.0)
    bishop_harvest  = trial.suggest_float('bishop_harvest', 1.0, 3.0)
    
    pawn_diet     = trial.suggest_float('pawn_diet', 0.3, 2.0)
    pawn_capacity = trial.suggest_float('pawn_capacity', 4.0, 10.0)
    pawn_harvest  = trial.suggest_float('pawn_harvest', 2.0, 5.0)
    
    # ── Identity constraints as penalties ──
    penalty = 0
    if queen_diet <= bishop_diet + 0.5: penalty += 10
    if bishop_diet <= pawn_diet + 0.3:  penalty += 10
    if pawn_harvest <= bishop_harvest + 0.3: penalty += 10
    if bishop_harvest <= queen_harvest + 0.3: penalty += 10
    if queen_capacity <= bishop_capacity + 1.0: penalty += 10
    
    profiles = {
        PieceType.QUEEN:  (queen_diet, queen_capacity, queen_harvest),
        PieceType.BISHOP: (bishop_diet, bishop_capacity, bishop_harvest),
        PieceType.PAWN:   (pawn_diet, pawn_capacity, pawn_harvest),
    }
    
    # ── Scenarios ──
    scenarios = [
        # (queens, bishops, pawns, target_pct, weight)
        (1, 0, 0,  30, 1.0),   # queen alone: should fail ~30%
        (1, 0, 3,  90, 2.0),   # barely makes it
        (1, 0, 5, 100, 1.0),   # easy
        (4, 0, 3,  40, 1.0),   # 4 queens understaffed: fail
        (4, 0, 8,  90, 2.0),   # 4 queens barely
        (1, 1, 2,  85, 2.0),   # bishop helps
    ]
    
    total_error = 0
    for nq, nb, np_, target, weight in scenarios:
        pct = run_scenario(profiles, nq, nb, np_)
        
        # Report intermediate for pruning
        trial.report(pct, step=scenarios.index((nq, nb, np_, target, weight)))
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        error = abs(pct - target)
        total_error += weight * error
    
    return total_error + penalty


def run_scenario(profiles, n_queens, n_bishops, n_pawns):
    """Run leapfrog with given piece stats, return range %."""
    myStuff = GameParts()
    myStuff.profile()
    fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
    food_tiers = fy.tiers
    grid, elevs = myStuff.grid, myStuff.terr.elevations
    countries = myStuff.terr.fields.get('country')
    
    myStuff.queenGuard.pieces.clear()
    piece_types = ([(PieceType.QUEEN, n_queens),
                    (PieceType.BISHOP, n_bishops),
                    (PieceType.PAWN, n_pawns)])
    
    for ptype, count in piece_types:
        for _ in range(count):
            p = myStuff.home.recruit(ptype, flag=myStuff.queenGuard.flag)
            diet, cap, harv = profiles[ptype]
            p.diet, p.food_capacity, p.harvest_strength = diet, cap, harv
            myStuff.queenGuard.pieces.append(p)
    
    myStuff.home.place_squad_spiral(myStuff.queenGuard, myStuff.terr)
    for p in myStuff.queenGuard.alive:
        p.food = p.food_capacity
    
    leader = myStuff.queenGuard.by_rank()[0]
    far_west = leader.location
    for _ in range(200):
        d = HexPosition.directions()[1]
        nb = grid.hexposition_to_index(d, far_west)
        if nb < 0 or nb >= len(elevs) or elevs[nb] < 1: break
        if countries is not None and countries[nb] < 0: break
        far_west = nb
    
    route = leader.pathfind(far_west, grid, elevs, countries)
    if not route: return 0.0
    
    # Run leapfrog (inline — same as demoSquad)
    pos, legs = 0, []
    for _ in range(50):
        if pos >= len(route) - 1: break
        sub = TroopPath(route.hexes[pos:],
                        sum(_move_cost(elevs, route.hexes[pos+k], route.hexes[pos+k+1])
                            for k in range(len(route.hexes[pos:]) - 1)))
        leg = myStuff.queenGuard.march_range(
            sub, grid, elevs, formation=None, countries=countries,
            food_tiers=food_tiers, min_food_tier=2, leader=leader)
        if not leg['feasible'] or leg['waypoint'] == 0: break
        
        wp = leg['hex_idx']
        best_fh, best_ft = wp, int(food_tiers[wp]) if wp < len(food_tiers) else 0
        for hp in HexPosition.origin().ring(1):
            nb = grid.hexposition_to_index(hp, wp)
            if 0 <= nb < len(food_tiers) and int(food_tiers[nb]) > best_ft:
                best_ft, best_fh = int(food_tiers[nb]), nb
        
        for p in myStuff.queenGuard.alive: p.location = best_fh
        cover = myStuff.queenGuard.feed_cover(food_tiers, objective='attack')
        
        for p in myStuff.queenGuard.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)
            if cover['feasible']:
                p.food = min(p.food_capacity, p.food + p.diet * 6)
        for p in myStuff.queenGuard.alive: p.location = wp
        
        legs.append(leg)
        pos += leg['waypoint']
    
    total = sum(l['waypoint'] for l in legs)
    return total / max(len(route) - 1, 1) * 100


In [ ]:
# ── Build map once, reuse everywhere ──
_map = GameParts()
_map.profile()
_fy = FoodYield(_map.terr, _map.basin); _fy.compute()
_food_tiers = _fy.tiers
_grid = _map.grid
_elevs = _map.terr.elevations
_countries = _map.terr.fields.get('country')

# Pre-compute the route from the home settlement
_home = _map.home
_route_cache = {}  # start_hex -> TroopPath

def _get_route(start_hex):
    """Cached route to far west from a given start."""
    if start_hex in _route_cache:
        return _route_cache[start_hex]
    far = start_hex
    for _ in range(200):
        d = HexPosition.directions()[1]
        nb = _grid.hexposition_to_index(d, far)
        if nb < 0 or nb >= len(_elevs) or _elevs[nb] < 1: break
        if _countries is not None and _countries[nb] < 0: break
        far = nb
    dummy = Piece(location=start_hex)
    route = dummy.pathfind(far, _grid, _elevs, _countries)
    _route_cache[start_hex] = route
    return route



In [ ]:
def run_scenario(profiles, n_queens, n_bishops, n_pawns,
                 recovery_turns=3):  # tighter recovery budget
    """Run leapfrog with given piece stats on the shared map."""
    squad = Squad(flag=_map.country.flag)
    squad.pieces = []
    
    for ptype, count in [(PieceType.QUEEN, n_queens),
                         (PieceType.BISHOP, n_bishops),
                         (PieceType.PAWN, n_pawns)]:
        for _ in range(count):
            p = _home.recruit(ptype, flag=squad.flag)
            diet, cap, harv = profiles[ptype]
            p.diet, p.food_capacity, p.harvest_strength = diet, cap, harv
            squad.pieces.append(p)
    
    _home.place_squad_spiral(squad, _map.terr)
    for p in squad.alive:
        p.food = p.food_capacity
    
    leader = squad.by_rank()[0]
    
    # Use farthest hex for a real challenge route
    route = _get_longest_route(leader.location)
    if not route: return 0.0
    
    pos, legs = 0, []
    for _ in range(50):
        if pos >= len(route) - 1: break
        sub_hexes = route.hexes[pos:]
        sub_cost = sum(_move_cost(_elevs, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)
        
        leg = squad.march_range(sub_path, _grid, _elevs,
                                formation=None, countries=_countries,
                                food_tiers=_food_tiers, min_food_tier=2,
                                leader=leader)
        if not leg['feasible'] or leg['waypoint'] == 0: break
        
        wp = leg['hex_idx']
        best_fh, best_ft = wp, int(_food_tiers[wp]) if wp < len(_food_tiers) else 0
        for hp in HexPosition.origin().ring(1):
            nb = _grid.hexposition_to_index(hp, wp)
            if 0 <= nb < len(_food_tiers) and int(_food_tiers[nb]) > best_ft:
                best_ft, best_fh = int(_food_tiers[nb]), nb
        
        for p in squad.alive: p.location = best_fh
        cover = squad.feed_cover(_food_tiers, objective='attack')
        
        for p in squad.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)
            if cover['feasible']:
                p.food = min(p.food_capacity, p.food + p.diet * recovery_turns)
        for p in squad.alive: p.location = wp
        
        legs.append(leg)
        pos += leg['waypoint']
    
    total = sum(l['waypoint'] for l in legs)
    return total / max(len(route) - 1, 1) * 100


def _get_longest_route(start):
    """Dijkstra flood to find the farthest reachable hex."""
    if 'longest' in _route_cache:
        return _route_cache['longest']
    import heapq
    pq = [(0.0, start)]
    best = {start: 0.0}
    visited = set()
    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited.add(cur)
        for nb in _grid.neighborsOf(cur):
            if nb in visited or nb in _grid.invalidRegion: continue
            if _elevs[nb] <= 0: continue
            if _countries is not None and _countries[nb] < 0: continue
            nc = cost + _move_cost(_elevs, cur, nb)
            if nc < best.get(nb, float('inf')):
                best[nb] = nc
                heapq.heappush(pq, (nc, nb))
    far = max(best, key=best.get)
    dummy = Piece(location=start, move_strength=5)
    route = dummy.pathfind(far, _grid, _elevs, _countries)
    _route_cache['longest'] = route
    print(f"Longest route: {len(route)} hexes, cost {best[far]:.0f}")
    return route


In [ ]:



def _get_longest_route(start):
    """Dijkstra flood to find the farthest reachable hex."""
    if 'longest' in _route_cache:
        return _route_cache['longest']
    import heapq
    pq = [(0.0, start)]
    best = {start: 0.0}
    visited = set()
    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited.add(cur)
        for nb in _grid.neighborsOf(cur):
            if nb in visited or nb in _grid.invalidRegion: continue
            if _elevs[nb] <= 0: continue
            if _countries is not None and _countries[nb] < 0: continue
            nc = cost + _move_cost(_elevs, cur, nb)
            if nc < best.get(nb, float('inf')):
                best[nb] = nc
                heapq.heappush(pq, (nc, nb))
    far = max(best, key=best.get)
    dummy = Piece(location=start, move_strength=5)
    route = dummy.pathfind(far, _grid, _elevs, _countries)
    _route_cache['longest'] = route
    print(f"Longest route: {len(route)} hexes, cost {best[far]:.0f}")
    return route


In [ ]:
def run_scenario(profiles, n_queens, n_bishops, n_pawns, recovery_turns=3):
    """Run leapfrog with surplus-sharing recovery."""
    squad = Squad(flag=_map.country.flag)
    squad.pieces = []
    
    for ptype, count in [(PieceType.QUEEN, n_queens),
                         (PieceType.BISHOP, n_bishops),
                         (PieceType.PAWN, n_pawns)]:
        for _ in range(count):
            p = _home.recruit(ptype, flag=squad.flag)
            diet, cap, harv = profiles[ptype][:3]
            p.diet, p.food_capacity, p.harvest_strength = diet, cap, harv
            if len(profiles[ptype]) > 3:
                p.move_strength = profiles[ptype][3]
            squad.pieces.append(p)
    
    _home.place_squad_spiral(squad, _map.terr)
    for p in squad.alive:
        p.food = p.food_capacity
    
    leader = squad.by_rank()[0]
    route = _get_longest_route(leader.location)
    if not route: return 0.0
    
    pos, legs = 0, []
    for _ in range(50):
        if pos >= len(route) - 1: break
        sub_hexes = route.hexes[pos:]
        sub_cost = sum(_move_cost(_elevs, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)
        
        leg = squad.march_range(sub_path, _grid, _elevs,
                                formation=None, countries=_countries,
                                food_tiers=_food_tiers, min_food_tier=2,
                                leader=leader)
        if not leg['feasible'] or leg['waypoint'] == 0: break
        
        wp = leg['hex_idx']
        
        # Find best food tier nearby
        best_ft = int(_food_tiers[wp]) if wp < len(_food_tiers) else 0
        best_fh = wp
        for hp in HexPosition.origin().ring(1):
            nb = _grid.hexposition_to_index(hp, wp)
            if 0 <= nb < len(_food_tiers) and int(_food_tiers[nb]) > best_ft:
                best_ft, best_fh = int(_food_tiers[nb]), nb
        
        # Burn food for travel
        for p in squad.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)
        
        # Recover via surplus-sharing
        for p in squad.alive: p.location = best_fh
        squad_recover(squad, best_ft, recovery_turns)
        for p in squad.alive: p.location = wp
        
        legs.append(leg)
        pos += leg['waypoint']
    
    total = sum(l['waypoint'] for l in legs)
    return total / max(len(route) - 1, 1) * 100


# Test with FoodProfile values directly
profiles = {
    PieceType.QUEEN:  (35, 40.0, 0.5, 3),
    PieceType.BISHOP: (1.5, 120.0, 1.0, 3),
    PieceType.PAWN:   (1.0,  4.0, 3.0, 2),
}

scenarios = [
    (1, 0, 0, "Queen alone"),
    (1, 1, 0, "Queen + Bishop"),
    (1, 0, 1, "Queen + pawn"),
    (1, 0, 2, "Queen + 2 pawn"),
    (1, 0, 3, "Queen + 3 Pawns (no bank)"),
    (1, 1, 2, "Queen + Bishop + 2 Pawns"),
    (1, 1, 4, "Queen + Bishop + 4 Pawns"),
    (1, 2, 3, "Queen + 2 Bishops + 3 Pawns"),
    (4, 1, 3, "4 Queens + 1 Bishop + 3 Pawns"),
    (4, 2, 8, "4 Queens + 2 Bishops + 8 Pawns"),
]

print(f"{'Scenario':40s} {'Range%':>7s}")
print("-" * 50)
for nq, nb, np_, label in scenarios:
    pct = run_scenario(profiles, nq, nb, np_)
    print(f"{label:40s} {pct:6.1f}%")


Should we tune?

can you rewrite the whole thing?

In [ ]:
# ─── Cell 1: FoodProfile ───────────────────────────────────────────────────

class FoodProfile:
    """Balanced food stats per piece type.
    
    Design intent:
      PAWN   = gatherer  — high harvest, tiny pockets
      BISHOP = transport — huge capacity, moderate harvest (the food bank)
      QUEEN  = consumer  — high diet, barely self-sufficient (needs escort)
    """
    PROFILES = {
        #                    diet   cap   harv  spd
        PieceType.PAWN:   (  1.0,   4.0,  5.0,  2),
        PieceType.BISHOP: (  1.5, 120.0,  1.0,  3),
        PieceType.ROOK:   (  2.5,  10.0,  2.0,  3),
        PieceType.KNIGHT: (  3.0,  10.0,  1.5,  4),
        PieceType.QUEEN:  ( 35.0,  40.0,  0.5,  3),
        PieceType.KING:   (  5.0,  12.0,  0.5,  2),
    }

    @classmethod
    def apply(cls, piece):
        diet, cap, harv, spd = cls.PROFILES.get(
            piece.piece_type, (1.0, 8.0, 2.0, 3))
        piece.diet             = diet
        piece.food_capacity    = cap
        piece.harvest_strength = harv
        piece.move_strength    = spd
        piece.food             = cap
        return piece

    @classmethod
    def summary(cls):
        print(f"{'Type':8s} {'Diet':>6s} {'Cap':>6s} {'Harv':>5s} "
              f"{'Spd':>4s} {'Turns':>6s} {'Net@T5':>7s}")
        print("-" * 50)
        for pt, (d, c, h, s) in cls.PROFILES.items():
            turns = c / d if d > 0 else float('inf')
            net   = h * 5 - d
            print(f"{pt.name:8s} {d:6.1f} {c:6.1f} {h:5.1f} "
                  f"{s:4d} {turns:6.1f} {net:+7.1f}")

FoodProfile.summary()


In [ ]:
# ─── Cell 2: squad_recover & pre_march_distribute ─────────────────────────

def pre_march_distribute(squad):
    """Before each leg, bishop tops up hungrier squad members.
    
    Donors keep at least 2 turns of food for themselves.
    Neediest (by food/capacity ratio) are filled first.
    """
    for p in sorted(squad.alive, key=lambda x: x.food / x.food_capacity):
        if p.food >= p.food_capacity:
            continue
        need = p.food_capacity - p.food
        # Bishops donate first, then anyone else richest-first
        donors = sorted(
            squad.alive,
            key=lambda d: (d.piece_type != PieceType.BISHOP, -d.food)
        )
        for donor in donors:
            if donor.id == p.id:
                continue
            reserve = donor.diet * 2          # keep 2 turns for self
            give    = min(need, max(0, donor.food - reserve))
            if give <= 0:
                continue
            p.food    += give
            donor.food -= give
            need       -= give
            if need <= 0:
                break


def squad_recover(squad, food_tier, recovery_turns=3):
    """Post-march harvest + surplus sharing.
    
    Phase 1 — each piece harvests and eats independently.
    Phase 2 — overflow goes to bishops (food bank), then others.
    Phase 3 — deficit pieces draw from bishops first, then richest.
    
    Returns (total_surplus, total_deficit) after all transfers.
    """
    surpluses = {}   # id → excess food that didn't fit
    deficits  = {}   # id → food still owed

    # Phase 1
    for p in squad.alive:
        production = p.harvest_strength * food_tier * recovery_turns
        consumed   = p.diet             *             recovery_turns
        new_food   = p.food + production - consumed

        if new_food > p.food_capacity:
            surpluses[p.id] = new_food - p.food_capacity
            p.food = p.food_capacity
        elif new_food < 0:
            deficits[p.id] = -new_food
            p.food = 0
        else:
            p.food = new_food

    total_surplus = sum(surpluses.values())
    total_deficit = sum(deficits.values())

    # Phase 2 — pour surplus into bishops first
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    others  = [p for p in squad.alive if p.piece_type != PieceType.BISHOP]

    for bank in sorted(bishops, key=lambda p: p.food):
        if total_surplus <= 0:
            break
        space         = bank.food_capacity - bank.food
        give          = min(space, total_surplus)
        bank.food    += give
        total_surplus -= give

    # Any remaining surplus spills into other pieces (least full first)
    for p in sorted(others, key=lambda p: p.food / max(p.food_capacity, 1)):
        if total_surplus <= 0:
            break
        space         = p.food_capacity - p.food
        give          = min(space, total_surplus)
        p.food       += give
        total_surplus -= give

    # Phase 3 — deficit pieces draw from bishops, then richest
    if total_deficit > 0:
        donors = sorted(
            squad.alive,
            key=lambda d: (d.piece_type != PieceType.BISHOP, -d.food)
        )
        for p in squad.alive:
            if p.id not in deficits:
                continue
            need = deficits[p.id]
            for donor in donors:
                if donor.id == p.id:
                    continue
                give       = min(need, max(0, donor.food - donor.diet))
                p.food    += give
                donor.food -= give
                need       -= give
                if need <= 0:
                    break

    return total_surplus, total_deficit


In [ ]:
# ─── Cell 3: run_scenario ─────────────────────────────────────────────────

def run_scenario(profiles, n_queens, n_bishops, n_pawns,
                 recovery_turns=3, verbose=False):
    """Leapfrog march with bishop-mediated food economy.
    
    profiles: dict of PieceType → (diet, capacity, harvest[, speed])
    Returns percentage of longest route completed (0–100).
    """
    squad = Squad(flag=_map.country.flag)
    squad.pieces = []

    for ptype, count in [(PieceType.QUEEN,  n_queens),
                         (PieceType.BISHOP, n_bishops),
                         (PieceType.PAWN,   n_pawns)]:
        stats = profiles.get(ptype)
        if stats is None or count == 0:
            continue
        diet, cap, harv = stats[:3]
        spd = stats[3] if len(stats) > 3 else 3
        for _ in range(count):
            p = _home.recruit(ptype, flag=squad.flag)
            p.diet, p.food_capacity, p.harvest_strength = diet, cap, harv
            p.move_strength = spd
            p.food = cap
            squad.pieces.append(p)

    _home.place_squad_spiral(squad, _map.terr)
    leader = squad.by_rank()[0]
    route  = _get_longest_route(leader.location)
    if not route:
        return 0.0

    pos, legs = 0, []

    for leg_num in range(50):
        if pos >= len(route) - 1:
            break

        # ── Pre-march: bishop tops up queen ──────────────────────
        pre_march_distribute(squad)

        # ── Build sub-path from current position ─────────────────
        sub_hexes = route.hexes[pos:]
        sub_cost  = sum(
            _move_cost(_elevs, sub_hexes[k], sub_hexes[k + 1])
            for k in range(len(sub_hexes) - 1)
        )
        sub_path = TroopPath(sub_hexes, sub_cost)

        # ── March ────────────────────────────────────────────────
        leg = squad.march_range(
            sub_path, _grid, _elevs,
            formation=None, countries=_countries,
            food_tiers=_food_tiers, min_food_tier=2,
            leader=leader,
        )
        if not leg['feasible'] or leg['waypoint'] == 0:
            break

        wp = leg['hex_idx']

        # ── Burn travel food ─────────────────────────────────────
        for p in squad.alive:
            tp    = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)

        # ── Find best nearby food tier ────────────────────────────
        best_ft, best_fh = (
            int(_food_tiers[wp]) if wp < len(_food_tiers) else 0), wp
        for hp in HexPosition.origin().ring(1):
            nb = _grid.hexposition_to_index(hp, wp)
            if 0 <= nb < len(_food_tiers) and int(_food_tiers[nb]) > best_ft:
                best_ft, best_fh = int(_food_tiers[nb]), nb

        # ── Recover at waypoint ───────────────────────────────────
        for p in squad.alive: p.location = best_fh
        surplus, deficit = squad_recover(squad, best_ft, recovery_turns)
        for p in squad.alive: p.location = wp

        if verbose:
            bishop_food = sum(p.food for p in squad.alive
                              if p.piece_type == PieceType.BISHOP)
            queen_food  = sum(p.food for p in squad.alive
                              if p.piece_type == PieceType.QUEEN)
            print(f"  Leg {leg_num+1:2d} | wp={leg['waypoint']:3d} "
                  f"| tier={best_ft} "
                  f"| bishop={bishop_food:6.1f} queen={queen_food:6.1f} "
                  f"| surplus={surplus:5.1f} deficit={deficit:5.1f}")

        legs.append(leg)
        pos += leg['waypoint']

    total = sum(l['waypoint'] for l in legs)
    return total / max(len(route) - 1, 1) * 100


In [ ]:
# ─── Cell 5: Optuna (only run if Cell 4 shows good spread) ────────────────

def balance_objective(trial):
    queen_diet     = trial.suggest_float('queen_diet',      10.0,  50.0)
    queen_capacity = trial.suggest_float('queen_capacity',  20.0,  60.0)
    queen_harvest  = trial.suggest_float('queen_harvest',    0.1,   1.0)

    bishop_diet     = trial.suggest_float('bishop_diet',    0.5,   3.0)
    bishop_capacity = trial.suggest_float('bishop_capacity',40.0, 200.0)
    bishop_harvest  = trial.suggest_float('bishop_harvest',  0.5,   3.0)

    pawn_diet     = trial.suggest_float('pawn_diet',    0.5,  3.0)
    pawn_capacity = trial.suggest_float('pawn_capacity', 2.0, 10.0)
    pawn_harvest  = trial.suggest_float('pawn_harvest',  2.0,  8.0)

    # Identity constraints — enforce role hierarchy
    penalty = 0
    if queen_diet     <= bishop_diet * 5:      penalty += 100  # queen much hungrier
    if bishop_capacity <= queen_capacity * 2:  penalty += 100  # bishop much bigger tank
    if pawn_harvest   <= bishop_harvest:        penalty += 50   # pawns harvest more
    if pawn_capacity  >= bishop_capacity * 0.2: penalty += 50  # pawns stay small

    profiles = {
        PieceType.QUEEN:  (queen_diet,  queen_capacity,  queen_harvest,  3),
        PieceType.BISHOP: (bishop_diet, bishop_capacity, bishop_harvest, 3),
        PieceType.PAWN:   (pawn_diet,   pawn_capacity,   pawn_harvest,   2),
    }

    # (Q, B, P, target%, weight)
    scenarios = [
        (1, 0, 0,   5,  1.0),  # queen alone: nearly dead
        (1, 1, 0,  20,  1.0),  # bishop helps but can't harvest
        (1, 0, 2,  65,  1.5),  # pawns harvest, no bank — capped
        (1, 0, 3,  75,  1.0),  # 3rd pawn wasted — should match 2-pawn
        (1, 1, 2,  90,  3.0),  # sweet spot — bishop + pawns
        (1, 1, 4, 100,  1.5),  # easy
        (4, 1, 4,  35,  2.0),  # 4 queens overwhelm 1 bishop
        (4, 2, 8,  90,  2.0),  # proper army
    ]

    cumulative_error = 0.0
    for step, (nq, nb, np_, target, weight) in enumerate(scenarios):
        pct   = run_scenario(profiles, nq, nb, np_)
        error = abs(pct - target) * weight
        cumulative_error += error
        trial.report(cumulative_error, step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return cumulative_error + penalty


storage = "sqlite:///balance.db"
study   = optuna.create_study(
    study_name="piece_balance_v5",
    storage=storage,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=3),
)
study.optimize(balance_objective, n_trials=300, show_progress_bar=True)

# ── Results ──────────────────────────────────────────────────────────────
print(f"\nBest error: {study.best_value:.1f}")
bp = study.best_params
best_profiles = {
    PieceType.QUEEN:  (bp['queen_diet'],  bp['queen_capacity'],  bp['queen_harvest'],  3),
    PieceType.BISHOP: (bp['bishop_diet'], bp['bishop_capacity'], bp['bishop_harvest'], 3),
    PieceType.PAWN:   (bp['pawn_diet'],   bp['pawn_capacity'],   bp['pawn_harvest'],   2),
}

print(f"\n{'Scenario':42s} {'Target':>7s} {'Actual':>7s} {'Error':>7s}")
print("-" * 62)
scenario_labels = [
    (1,0,0, 5,  "Queen alone"),
    (1,1,0, 20, "Queen + Bishop"),
    (1,0,2, 65, "Queen + 2 Pawns"),
    (1,0,3, 75, "Queen + 3 Pawns (no bank)"),
    (1,1,2, 90, "Queen + Bishop + 2 Pawns"),
    (1,1,4,100, "Queen + Bishop + 4 Pawns"),
    (4,1,4, 35, "4 Queens + 1 Bishop + 4 Pawns"),
    (4,2,8, 90, "4 Queens + 2 Bishops + 8 Pawns"),
]
for nq, nb, np_, target, label in scenario_labels:
    pct = run_scenario(best_profiles, nq, nb, np_)
    print(f"{label:42s} {target:6.0f}% {pct:6.1f}% {abs(pct-target):6.1f}")


Those look great

Lets call them design values. can you build the food profile

In [ ]:
class FoodProfile:
    """Tuned food stats per piece type.

    Design intent & roles:
      PAWN   = gatherer    — harvests for others, tiny personal reserves
      BISHOP = food bank   — massive capacity, moderate harvest, bankrolls the queen
      QUEEN  = consumer    — high diet, barely self-sufficient, needs escort
      ROOK   = heavy foot  — moderate diet, self-sustaining on good terrain
      KNIGHT = fast raider — higher diet, self-sustaining, speed advantage
      KING   = precious    — low harvest, moderate reserves, must be protected

    Tuned via Optuna (300 trials) against scenario table:
      Queen alone          →  ~5% range
      Queen + Bishop       → ~22% range
      Queen + 2 Pawns      → ~63% range
      Queen + 3 Pawns      → ~75% range  (bank-less ceiling)
      Queen + Bishop + 2P  → ~90% range  (sweet spot)
      Queen + Bishop + 4P  → 100% range
    """

    PROFILES = {
        #                        diet    cap   harv  spd
        PieceType.PAWN:   (      2.7,    8.7,  2.06,  2),
        PieceType.BISHOP: (      1.9,  178.7,  0.84,  3),
        PieceType.ROOK:   (      3.5,   12.0,  1.8,   3),
        PieceType.KNIGHT: (      4.0,   10.0,  1.5,   4),
        PieceType.QUEEN:  (     27.9,   59.0,  0.48,  3),
        PieceType.KING:   (      5.0,   14.0,  0.5,   2),
    }

    @classmethod
    def apply(cls, piece):
        """Apply profile stats to a piece in place. Returns the piece."""
        diet, cap, harv, spd = cls.PROFILES.get(
            piece.piece_type, (1.0, 8.0, 2.0, 3))
        piece.diet             = diet
        piece.food_capacity    = cap
        piece.harvest_strength = harv
        piece.move_strength    = spd
        piece.food             = cap
        return piece

    @classmethod
    def summary(cls):
        """Print a human-readable table of all profiles."""
        print(f"{'Type':8s} {'Diet':>6s} {'Cap':>7s} {'Harv':>5s} "
              f"{'Spd':>4s} {'Turns':>6s} {'Net@T5':>8s} {'Net@T7':>8s}")
        print("─" * 60)
        for pt, (d, c, h, s) in cls.PROFILES.items():
            turns  = c / d if d > 0 else float('inf')
            net_t5 = h * 5 - d
            net_t7 = h * 7 - d
            print(f"{pt.name:8s} {d:6.1f} {c:7.1f} {h:5.2f} "
                  f"{s:4d} {turns:6.1f} {net_t5:+8.2f} {net_t7:+8.2f}")
        print()
        print("Net@T5/T7 = harvest surplus per turn on tier-5/7 tile.")
        print("Positive = self-sustaining. Negative = needs support.")

FoodProfile.summary()


I know for some functions like food_cover I have multiple versions. some of the older stuff isn't working. what are the best versions of the multiple functions

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — there's definitely some drift. Here's a map of the duplicates and which version to keep:

**`FoodProfile`** — 3 versions
- ❌ Note cell `_d1c3c334` (reference only, not executable)
- ❌ Cell `_eb48061b` (intermediate values: queen_diet=35, pawn_harv=5.0)
- ✅ **Cell `_a51642c9`** — the Optuna-tuned design values (queen 27.9/59.0/0.48, bishop 1.9/178.7/0.84, pawn 2.7/8.7/2.06)

**`run_scenario`** — 3 versions
- ❌ Cell `_47f2a5b3` (inside balance_objective, old recovery)
- ❌ Cell `_54404c50` (uses old per-piece recovery, no `pre_march_distribute`)
- ✅ **Cell `_d35ec0dd`** — calls `pre_march_distribute` before each leg + `squad_recover` after

**`squad_recover`** — 2 versions
- ❌ Cell `_4c7d63a2` (first draft)
- ✅ **Cell `_0542376a`** — cleaner, same logic but better structured comments. Also contains `pre_march_distribute` which only exists here.

**`balance_objective`** — 2 versions
- ❌ Cell `_47f2a5b3` (narrow param ranges, wrong identity constraints)
- ✅ **Cell `_bfee8097`** — wider ranges, queen_diet 10–50, bishop_capacity 40–200, proper hierarchy penalties

**`_get_longest_route`** — 2 identical copies
- ❌ Cell `_54404c50` (embedded in the old run_scenario cell)
- ✅ **Cell `_02ea8bbe`** — standalone, keep this one

**`_FREE` / `_COSTS_ONE`** — 2 versions
- ✅ **Cell `_d5185eee`** — includes DEFEND and SETTLE: `{PAUSE, DEFEND, HARVEST, GIVE, SETTLE}`
- ❌ Cell `_d3b6905c` — missing DEFEND and SETTLE, will silently skip those instructions

**`SimEventType`** — only in cell `_d5185eee` ✅ (includes BUMPED)

**Functions with only one version** (all fine):
- `FoodSimulator` → cell `_29b1dbba`
- `_try_bump` → cell `_7ffddf90`
- `_try_forward` → cell `_f385d473` (with bump integration)
- `demoSquad` → single version
- `leapfrog_plan`, `march_range`, `march_analysis` → single versions each

The biggest landmine is that `_COSTS_ONE` cell — if it runs after the full definition, it silently clobbers it and DEFEND/SETTLE become free no-ops. I'd delete cell `_d3b6905c` entirely since `_d5185eee` already defines both sets.

is this the better squad recover

```
def squad_recover(squad, food_tier, recovery_turns=3):
    """Post-march harvest + surplus sharing.
    
    Phase 1 — each piece harvests and eats independently.
    Phase 2 — overflow goes to bishops (food bank), then others.
    Phase 3 — deficit pieces draw from bishops first, then richest.
    
    Returns (total_surplus, total_deficit) after all transfers.
    """
    surpluses = {}   # id → excess food that didn't fit
    deficits  = {}   # id → food still owed

    # Phase 1
    for p in squad.alive:
        production = p.harvest_strength * food_tier * recovery_turns
        consumed   = p.diet             *             recovery_turns
        new_food   = p.food + production - consumed

        if new_food > p.food_capacity:
            surpluses[p.id] = new_food - p.food_capacity
            p.food = p.food_capacity
        elif new_food < 0:
            deficits[p.id] = -new_food
            p.food = 0
        else:
            p.food = new_food

    total_surplus = sum(surpluses.values())
    total_deficit = sum(deficits.values())

    # Phase 2 — pour surplus into bishops first
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    others  = [p for p in squad.alive if p.piece_type != PieceType.BISHOP]

    for bank in sorted(bishops, key=lambda p: p.food):
        if total_surplus <= 0:
            break
        space         = bank.food_capacity - bank.food
        give          = min(space, total_surplus)
        bank.food    += give
        total_surplus -= give

    # Any remaining surplus spills into other pieces (least full first)
    for p in sorted(others, key=lambda p: p.food / max(p.food_capacity, 1)):
        if total_surplus <= 0:
            break
        space         = p.food_capacity - p.food
        give          = min(space, total_surplus)
        p.food       += give
        total_surplus -= give

    # Phase 3 — deficit pieces draw from bishops, then richest
    if total_deficit > 0:
        donors = sorted(
            squad.alive,
            key=lambda d: (d.piece_type != PieceType.BISHOP, -d.food)
        )
        for p in squad.alive:
            if p.id not in deficits:
                continue
            need = deficits[p.id]
            for donor in donors:
                if donor.id == p.id:
                    continue
                give       = min(need, max(0, donor.food - donor.diet))
                p.food    += give
                donor.food -= give
                need       -= give
                if need <= 0:
                    break

    return total_surplus, total_deficit
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Yes, that's the better one. It's from cell `_0542376a` and it's better than the earlier version in cell `_4c7d63a2` for two reasons:

1. **Phase 2 donor ordering** — it sorts bishops by least-full first (`key=lambda p: p.food`), so it fills the emptiest bank first. The older version just iterated them in arbitrary order.

2. **Phase 3 donor floor** — it uses `donor.food - donor.diet` as the floor (keeps 1 turn of food), while the older version also does this but with a less clear donor sort. This version explicitly sorts donors as `(d.piece_type != PieceType.BISHOP, -d.food)` — bishops first, then richest. The older version just did `sorted(squad.alive, key=lambda p: -p.food)` which means a rich pawn could get drained before the bishop.

The only thing this version is missing is `pre_march_distribute` which lives in the same cell. Make sure you keep that too — it's what lets the bishop top up the queen *before* each leg so `march_range`'s per-piece food check doesn't reject legs the squad could actually survive.